# Research data supporting "Practical and reliable density functionals for heterogeneous transition-metal catalysis"

This notebook accompanies our paper: **Practical and reliable density functionals for heterogeneous transition-metal catalysis**. It can be found on GitHub at https://github.com/benshi97/Data_NSCDFT_for_Metals and explored interactively on [Colab](https://colab.research.google.com/github/benshi97/Data_NSCDFT_for_Metals/blob/main/analyse.ipynb).

### Abstract

Density functional theory (DFT) has been critical towards our current atomistic understanding of catalysis on transition-metal surfaces.
It has opened new paradigms in rational catalyst design, predicting fundamental properties, like the adsorption energy and reaction barriers, linked to catalytic performance.
However, such applications depend sensitively on the predictive accuracy of DFT and achieving experimental-level reliability, so-called transition-metal chemical accuracy (13 kJ/mol), remains challenging for present density functional approximations (DFAs) or even methods beyond DFT.
We introduce a new framework for designing DFAs tailored towards studying molecules adsorbed on transition-metal surfaces, building upon recent developments in non-self-consistent DFAs.
We propose two functionals within this framework, demonstrating that transition-metal chemical accuracy can be achieved across a diverse set of 39 adsorption reactions while delivering consistent performance for 17 barrier heights.
Moreover, we show that these non-self-consistent DFAs address qualitative failures that challenge current self-consistent DFAs, such as CO adsorption on Pt(111) and graphene on Ni(111).
The resulting functionals are computationally practical and compatible with existing DFT codes, with scripts and workflows provided to use them.
Looking ahead, this framework establishes a path toward developing more accurate and sophisticated DFAs for computational heterogeneous catalysis and beyond.


## Table of Contents

- [Practicality of Framework](#practicality-of-framework)
- [CE39 Dataset](#ce39-dataset)
- [Adsorption Configuration](#adsorption-configuration)
- [Barrier Heights](#barrier-heights)


In [91]:
# Check if we are in Google Colab environment
try:
    import google.colab
    IN_COLAB = True
    usetex = False
    import os
    import subprocess
    import sys
except:
    import os
    IN_COLAB = False
    if os.path.expanduser('~') == '/Users/bshi':
        usetex = True
    else:
        usetex = False

if usetex:
    def textbf(text):
        return r'\textbf{' + text + '}'
else:
    def textbf(text):
        return text

# If in Google Colab, install the necessary data and set up the necessary environment
if IN_COLAB == True:
    repo_url = "https://github.com/benshi97/Data_NSCDFT_for_Metals"

    # Clone the repository
    %cd /content
    !rm -rf /content/Data_NSCDFT_for_Metals
    !git clone {repo_url}
    %cd /content/Data_NSCDFT_for_Metals

replot_analysis = False

# Import necessary functions
from Scripts.analysis_scripts import *
from Scripts.plot_scripts import *

if usetex == True:
    textrue_import()
else:
    texfalse_import()

dft_xc_list = [
    "01_PBE",
    "02_PBEsol",
    "03_RPBE",
    "04_revPBE",
    "05_BLYP",
    "06_r2SCAN",
    "07_MS2",
    "08_revTPSS",
    "09_PBED3",
    "10_PBEDDsC",
    "11_RPBED3",
    "12_optPBEvdW",
    "13_revvdWDF2",
    "14_r2SCANrVV10"
]

method_dict = {
    '01_PBE': {'label': 'PBE', 'color': 'orange'},
    '02_PBEsol': {'label': 'PBEsol', 'color': 'purple'},
    '03_RPBE': {'label': 'RPBE', 'color': 'cyan'},
    '04_revPBE': {'label': 'revPBE', 'color': 'magenta'},
    '05_BLYP': {'label': 'BLYP', 'color': 'lime'},
    '06_r2SCAN': {'label': r'r$^2$SCAN', 'color': 'pink'},
    '07_MS2': {'label': 'MS2', 'color': 'teal'},
    '08_revTPSS': {'label': 'revTPSS', 'color': 'lavendar'},
    '09_PBED3': {'label': 'PBE-D3', 'color': 'brown'},
    '10_PBEDDsC': {'label': 'PBE-DdsC', 'color': 'beige'},
    '11_RPBED3': {'label': 'RPBE-D3', 'color': 'maroon'},
    '12_optPBEvdW': {'label': 'optPBE-vdW', 'color': 'apricot'},
    '13_revvdWDF2': {'label': 'rev-vdW-DF2', 'color': 'navy'},
    '14_r2SCANrVV10': {'label': r'r$^2$SCAN+rVV10', 'color': 'olive'},
    'RPA': {'label': 'RPA@PBE', 'color': 'blue'},
    'BEEFvdW': {'label': 'BEEF-vdW', 'color': 'red'},
    'hBEEFvdW': {'label': 'hBEEF-vdW', 'color': 'green'},
    'dhBEEFvdW': {'label': 'dhBEEF-vdW', 'color': 'yellow'},
}


# Practicality of framework

## Gas phase energy

In [92]:
method_comparison_list = ['EXX_0','EXX_1','EXX_2','EXX_3']

exx_kpoint_energy = { kpoint: {system : {vac: {method: 0 for method in method_comparison_list} for vac in [4,6,8,10,12,14,16,18]} for system in ['CO', 'H2O']} for kpoint in [1,2,3] }

for kpoint in [1,2,3]:
    for system in ['CO', 'H2O']:
        for vac in [4,6,8,10,12,14,16,18]:
            for method_idx, method in enumerate(method_comparison_list):
                exx_kpoint_energy[kpoint][system][vac][method] = get_energy(f'Data/Convergence/Gas/EXX/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_03_{method_idx}.gz', code='vasp')

# Plot for kpoint convergence of EXX contribution for CO and H2O

# Plot for CO

fig, axs = plt.subplots(1,4, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for method_idx, method in enumerate(method_comparison_list):
    axs[method_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[method_idx].scatter([vac for vac in [4,6,8,10,12,14,16,18]], [(exx_kpoint_energy[1]['CO'][vac][method] - exx_kpoint_energy[1]['CO'][18][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16,18]], label=r'$1\times1\times1$', color='black', marker='x')
    axs[method_idx].scatter([vac for vac in [4,6,8,10,12,14,16,18]], [(exx_kpoint_energy[2]['CO'][vac][method] - exx_kpoint_energy[2]['CO'][18][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16,18]], label=r'$2\times2\times2$', color='black', marker='^', facecolors='none')
    axs[method_idx].scatter([vac for vac in [4,6,8,10,12,14,16,18]], [(exx_kpoint_energy[3]['CO'][vac][method] - exx_kpoint_energy[3]['CO'][18][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16,18]], label=r'$3\times3\times3$', color='black', marker='s', facecolors='none')

    axs[method_idx].set_title(r'$\omega=\,$' + f'0.{method_idx}$\,$' + r'\AA{}$^{-1}$')
    axs[method_idx].set_xlabel(r'Vacuum size (\AA{})')

axs[0].set_ylim([-20,20])
axs[0].set_xlim([2,20])
axs[0].set_xticks([2,8,14,20])
axs[1].legend()
fig.suptitle('EXX energy convergence with vacuum size for CO molecule')

axs[0].set_ylabel(r'EXX energy w.r.t. 18\AA{} (kJ/mol)')

plt.savefig('Figures/SI_Figure-EXX_KPOINTS_CO_Convergence.png')

# Plot for H2O

fig, axs = plt.subplots(1,4, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for method_idx, method in enumerate(method_comparison_list):
    axs[method_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[method_idx].scatter([vac for vac in [4,6,8,10,12,14,16,18]], [(exx_kpoint_energy[1]['H2O'][vac][method] - exx_kpoint_energy[1]['H2O'][18][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16,18]], label=r'$1\times1\times1$', color='black', marker='x')
    axs[method_idx].scatter([vac for vac in [4,6,8,10,12,14,16,18]], [(exx_kpoint_energy[2]['H2O'][vac][method] - exx_kpoint_energy[2]['H2O'][18][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16,18]], label=r'$2\times2\times2$', color='black', marker='^', facecolors='none')
    axs[method_idx].scatter([vac for vac in [4,6,8,10,12,14,16,18]], [(exx_kpoint_energy[3]['H2O'][vac][method] - exx_kpoint_energy[3]['H2O'][18][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16,18]], label=r'$3\times3\times3$', color='black', marker='s', facecolors='none')

    axs[method_idx].set_title(r'$\omega=\,$' + f'0.{method_idx}$\,$' + r'\AA{}$^{-1}$')
    axs[method_idx].set_xlabel(r'Vacuum size (\AA{})')

axs[0].set_ylim([-20,20])
axs[0].set_xlim([2,20])
axs[0].set_xticks([2,8,14,20])
axs[1].legend()
fig.suptitle('EXX energy convergence with vacuum size for H2O molecule')

axs[0].set_ylabel(r'EXX energy w.r.t. 18\AA{} (kJ/mol)')

plt.savefig('Figures/SI_Figure-EXX_KPOINTS_H2O_Convergence.png')

In [93]:
rpac_kpoint_energy = { 
    1: {system : {vac: 0 for vac in [4,6,8,10,12,14,16]} for system in ['CO', 'H2O'] },
    2: {system : {vac: 0 for vac in [4,6,8,10]} for system in ['CO', 'H2O'] },
    3: {system : {vac: 0 for vac in [4,6,8,10]} for system in ['CO', 'H2O'] }
}

for kpoint in [1,2,3]:
    for system in ['CO', 'H2O']:
        for vac in [4,6,8,10,12,14,16]:
            if vac > 10 and kpoint > 1:
                continue
            rpac_kpoint_energy[kpoint][system][vac] = get_energy(f'Data/Convergence/Gas/RPA/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_04_RPA.gz', code='vasp_rpa')

# Plot for kpoint convergence of RPA correlation contribution for CO and H2O

# Plot for CO and H2O

fig, axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[system_idx].scatter([vac for vac in [4,6,8,10,12,14,16]], [(rpac_kpoint_energy[1][system][vac] - rpac_kpoint_energy[1][system][10]) * 1000 / kjmol_to_meV for vac in [4,6,8,10,12,14,16]], label=r'$1\times1\times1$', color='black', marker='x')
    axs[system_idx].scatter([vac for vac in [4,6,8,10]], [(rpac_kpoint_energy[2][system][vac] - rpac_kpoint_energy[2][system][10]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$2\times2\times2$', color='black', marker='^', facecolors='none')
    axs[system_idx].scatter([vac for vac in [4,6,8,10]], [(rpac_kpoint_energy[3][system][vac] - rpac_kpoint_energy[3][system][10]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$3\times3\times3$', color='black', marker='s', facecolors='none')

    axs[system_idx].set_title(system)
    axs[system_idx].set_xlabel(r'Vacuum size (\AA{})')

axs[0].set_ylim([-20,20])
axs[0].set_xlim([2,18])
axs[0].set_xticks([2,4,6,8,10,12,14,16,18])
axs[1].legend()
fig.suptitle('RPA correlation energy convergence with vacuum size')
axs[0].set_ylabel(r'RPA correlation energy w.r.t. 10\AA{} (kJ/mol)')
plt.savefig('Figures/SI_Figure-RPA_KPOINTS_Convergence.png')

In [94]:
total_method_comparison_list = ['RPA', 'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW']

total_kpoint_energy = { kpoint: {system : {vac: {method: 0 for method in total_method_comparison_list} for vac in [4,6,8,10]} for system in ['CO', 'H2O']} for kpoint in [1,2,3] }

for kpoint in [1,2,3]:
    for system in ['CO', 'H2O']:
        for vac in [4,6,8,10]:
            # Start with RPA
            total_kpoint_energy[kpoint][system][vac]['RPA'] = get_energy(f'Data/Convergence/Gas/RPA/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_04_RPA.gz', code='vasp_rpa') + get_energy(f'Data/Convergence/Gas/RPA/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_02_EXX.gz', code='vasp')
            # Then BEEF-vdW
            total_kpoint_energy[kpoint][system][vac]['BEEFvdW'] = get_energy(f'Data/Convergence/Gas/EXX/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_00.gz', code='vasp')
            # Then hBEEF-vdW
            calc_energy_dict = {
                '00': 0,
                '01': 0,
                '02': 0,
                '03_3': 0,
            }
            # hBEEF part
            for calc_type in calc_energy_dict:
                calc_energy_dict[calc_type] = get_energy(f'Data/Convergence/Gas/EXX/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_{calc_type}.gz', code='vasp')

            beefx_energy = calc_energy_dict['02']
            exx_energy = calc_energy_dict['03_3']
            nlc_energy = calc_energy_dict['00'] - calc_energy_dict['01']
            beefc_energy = calc_energy_dict['01'] - calc_energy_dict['02']
            rpac_energy = get_energy(f'Data/Convergence/Gas/RPA/{system}/KPOINTS_{kpoint}/{vac}/OUTCAR_04_RPA.gz', code='vasp_rpa')
            dh_x_frac = 0.25
            dh_c_frac = 0.15
            total_kpoint_energy[kpoint][system][vac]['dhBEEFvdW'] = dh_x_frac * exx_energy + (1-dh_x_frac)*beefx_energy + dh_c_frac*rpac_energy + (1-dh_c_frac)*beefc_energy + nlc_energy
            h_x_frac = 0.175
            total_kpoint_energy[kpoint][system][vac]['hBEEFvdW'] = h_x_frac * exx_energy + (1-h_x_frac)*beefx_energy + beefc_energy + nlc_energy

# Plot for kpoint convergence of total energy for CO and H2O

# Plot for CO

fig, axs = plt.subplots(1,4, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for method_idx, method in enumerate(total_method_comparison_list):
    axs[method_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[1]['CO'][vac][method] - total_kpoint_energy[1]['CO'][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$1\times1\times1$', color='black', marker='x')
    axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[2]['CO'][vac][method] - total_kpoint_energy[2]['CO'][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$2\times2\times2$', color='black', marker='^', facecolors='none')
    axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[3]['CO'][vac][method] - total_kpoint_energy[3]['CO'][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$3\times3\times3$', color='black', marker='s', facecolors='none')
    if method == 'RPA':
        axs[method_idx].set_title('RPA')
    else:
        axs[method_idx].set_title(method_dict[method]['label'])
    axs[method_idx].set_xlabel(r'Vacuum size (\AA{})')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([2,12])
axs[0].set_xticks([2,4,6,8,10,12])
axs[1].legend()
fig.suptitle('Total energy convergence with vacuum size for CO molecule')
axs[0].set_ylabel(r'Total energy w.r.t. 10\AA{} (kJ/mol)')
plt.savefig('Figures/SI_Figure-Total_KPOINTS_CO_Convergence.png')

# Plot for H2O

fig, axs = plt.subplots(1,4, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for method_idx, method in enumerate(total_method_comparison_list):
    axs[method_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[1]['H2O'][vac][method] - total_kpoint_energy[1]['H2O'][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$1\times1\times1$', color='black', marker='x')
    axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[2]['H2O'][vac][method] - total_kpoint_energy[2]['H2O'][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$2\times2\times2$', color='black', marker='^', facecolors='none')
    axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[3]['H2O'][vac][method] - total_kpoint_energy[3]['H2O'][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$3\times3\times3$', color='black', marker='s', facecolors='none')
    if method == 'RPA':
        axs[method_idx].set_title('RPA')
    else:
        axs[method_idx].set_title(method_dict[method]['label'])
    axs[method_idx].set_xlabel(r'Vacuum size (\AA{})')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([2,12])
axs[0].set_xticks([2,4,6,8,10,12])
axs[1].legend()
fig.suptitle('Total energy convergence with vacuum size for H2O molecule')
axs[0].set_ylabel(r'Total energy w.r.t. 10\AA{} (kJ/mol)')
plt.savefig('Figures/SI_Figure-Total_KPOINTS_H2O_Convergence.png')

# Highlight the faster convergence of the total energy with vacuum size when using BEEF-vdW correction for hBEEF-vdW and dhBEEF-vdW

# Plot for CO and H2O

kpoint_error_convergence = { kpoint: {system : {vac: {method: 0 for method in ['RPA','hBEEFvdW','dhBEEFvdW','RPA+D','hBEEFvdW+D','dhBEEFvdW+D']} for vac in [4,6,8,10]} for system in ['CO', 'H2O']} for kpoint in [1,2,3] }

for system in ['CO','H2O']:
    for method_idx, method in enumerate(['RPA','hBEEFvdW', 'dhBEEFvdW']):
        for kpoint in [1,2,3]:
            for vac in [4,6,8,10]:
                kpoint_error_convergence[kpoint][system][vac][method] = total_kpoint_energy[kpoint][system][vac][method] - total_kpoint_energy[3][system][10][method]

for system in ['CO','H2O']:

    fig, axs = plt.subplots(1,3, figsize=(5,4), dpi=450, constrained_layout=True, sharey='row', sharex=True)

    for method_idx, method in enumerate(['RPA','hBEEFvdW', 'dhBEEFvdW']):
        axs[method_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
        axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[1][system][vac][method] - total_kpoint_energy[3][system][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$1\times1\times1$', color='black', marker='x')
        axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[2][system][vac][method] - total_kpoint_energy[3][system][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$2\times2\times2$', color='black', marker='^', facecolors='none')
        axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[3][system][vac][method] - total_kpoint_energy[3][system][10][method]) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$3\times3\times3$', color='black', marker='s', facecolors='none')
        axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[1][system][vac][method] - total_kpoint_energy[3][system][10][method] + total_kpoint_energy[3][system][10]['BEEFvdW'] - total_kpoint_energy[1][system][vac]['BEEFvdW']) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$1\times1\times1$ $+\Delta_\text{vac}$', color='red', marker='x')
        axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[2][system][vac][method] - total_kpoint_energy[3][system][10][method]  + total_kpoint_energy[3][system][10]['BEEFvdW'] - total_kpoint_energy[2][system][vac]['BEEFvdW']) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$2\times2\times2$ $+\Delta_\text{vac}$', color='red', marker='^', facecolors='none')
        axs[method_idx].scatter([vac for vac in [4,6,8,10]], [(total_kpoint_energy[3][system][vac][method] - total_kpoint_energy[3][system][10][method]  + total_kpoint_energy[3][system][10]['BEEFvdW'] - total_kpoint_energy[3][system][vac]['BEEFvdW']) * 1000 / kjmol_to_meV for vac in [4,6,8,10]], label=r'$3\times3\times3$ $+\Delta_\text{vac}$', color='red', marker='s', facecolors='none')
        if method == 'RPA':
            axs[method_idx].set_title('RPA')
        else:
            axs[method_idx].set_title(method_dict[method]['label'])
        axs[method_idx].set_xlabel(r'Vacuum size (\AA{})') 

    axs[0].set_ylim([-250,50])
    axs[0].set_xlim([2,12])
    axs[0].set_xticks([2,4,6,8,10,12])
    axs[1].legend()

    plt.savefig(f'Figures/SI_Figure-Total_Delta_{system}_Convergence.png')

## Adsorption energy

In [95]:
# Convergence of BEEF, EXX and RPAc and total
kpoint_conv_eads_dict = {system: {kpoint: {method: 0 for method in ['EXX','EXX_w_0','EXX_w_1','EXX_w_2','EXX_w_3','RPAc','RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW']} for kpoint in range(1,15)} for system in ['CO', 'H2O']}

for kpoint in range(1,15):
    for system in ['CO', 'H2O']:
        kpoint_conv_eads_dict[system][kpoint]['EXX'] = get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}_Pt/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/Pt/OUTCAR_02_EXX.gz', code='vasp')
        kpoint_conv_eads_dict[system][kpoint]['RPAc'] = get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}_Pt/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/Pt/OUTCAR_04_RPA.gz', code='vasp_rpa')
        kpoint_conv_eads_dict[system][kpoint]['RPA'] = kpoint_conv_eads_dict[system][kpoint]['EXX'] + kpoint_conv_eads_dict[system][kpoint]['RPAc']
        kpoint_conv_eads_dict[system][kpoint]['BEEFvdW'] = get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}_Pt/OUTCAR_00.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}/OUTCAR_00.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/Pt/OUTCAR_00.gz', code='vasp')
        for w_idx in ['0','1','2','3']:
            kpoint_conv_eads_dict[system][kpoint][f'EXX_w_{w_idx}'] = get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}_Pt/OUTCAR_03_{w_idx}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}/OUTCAR_03_{w_idx}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/Pt/OUTCAR_03_{w_idx}.gz', code='vasp')
        # hBEEF and dhBEEF
        calc_energy_dict = {
            '00': 0,
            '01': 0,
            '02': 0,
            '03_3': 0,
        }
        for calc_type in calc_energy_dict:
            calc_energy_dict[calc_type] = get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}_Pt/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/Pt/OUTCAR_{calc_type}.gz', code='vasp')

        beefx_energy = calc_energy_dict['02']
        exx_energy = calc_energy_dict['03_3']
        nlc_energy = calc_energy_dict['00'] - calc_energy_dict['01']
        beefc_energy = calc_energy_dict['01'] - calc_energy_dict['02']
        rpac_energy = kpoint_conv_eads_dict[system][kpoint]['RPAc']
        dh_x_frac = 0.25
        dh_c_frac = 0.15
        kpoint_conv_eads_dict[system][kpoint]['dhBEEFvdW'] = dh_x_frac * exx_energy + (1-dh_x_frac)*beefx_energy + dh_c_frac*rpac_energy + (1-dh_c_frac)*beefc_energy + nlc_energy
        h_x_frac = 0.175
        kpoint_conv_eads_dict[system][kpoint]['hBEEFvdW'] = h_x_frac * exx_energy + (1-h_x_frac)*beefx_energy + beefc_energy + nlc_energy

# Plot convergence for EXX with different range-separation parameters
fig, axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    color_list = ['purple', 'cyan', 'magenta', 'lime']
    marker_list = ['x', '>', 's', 'd']
    facecolor_list = ['purple', 'none', 'none', 'none']
    for w_idx in [0,1,2,3]:
        axs[system_idx].plot([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint][f'EXX_w_{str(w_idx)}'] - kpoint_conv_eads_dict[system][14][f'EXX_w_{w_idx}']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label=r'EXX ($\omega=0.' + str(w_idx) + r'\,$\AA{}$^{-1}$)', marker=marker_list[w_idx],linestyle='--',linewidth=0.7,color=color_dict[color_list[w_idx]],markerfacecolor=facecolor_list[w_idx])

    # Add RPA correlation
    axs[system_idx].plot([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['RPAc'] - kpoint_conv_eads_dict[system][14]['RPAc']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='RPA correlation', color='green', marker='^',markerfacecolor='none',linestyle='--',linewidth=0.7)

    axs[system_idx].set_title(f'{system} on Pt(111)')
    axs[system_idx].set_xlabel(r'$k$-point grid ($M \times M \times 1$)')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([0,15])
axs[0].set_xticks([0,2,4,6,8,10,12,14])
axs[1].legend(ncols=2,fontsize=8)
fig.suptitle(r'$E_\text{ads}$ $k$-points convergence for EXX with different $\omega$')
axs[0].set_ylabel(r'$E_\text{ads}$ w.r.t. $14 \times 14 \times 1$ $k$-point grid (kJ/mol)')
plt.savefig('Figures/SI_Figure-Eads_EXX_KPOINTS_Convergence.png')


# Plot for CO and H2O
fig,axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    # Plot hBEEFvdW and dhBEEFvdW
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['hBEEFvdW'] - kpoint_conv_eads_dict[system][14]['hBEEFvdW']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='hBEEF-vdW', color=color_dict[method_dict['hBEEFvdW']['color']], marker='^',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['dhBEEFvdW'] - kpoint_conv_eads_dict[system][14]['dhBEEFvdW']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='dhBEEF-vdW', color=color_dict[method_dict['dhBEEFvdW']['color']], marker='v',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['RPA'] - kpoint_conv_eads_dict[system][14]['RPA']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='Total RPA', color=color_dict[method_dict['RPA']['color']], marker='s',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['BEEFvdW'] - kpoint_conv_eads_dict[system][14]['BEEFvdW']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='BEEF-vdW', color=color_dict[method_dict['BEEFvdW']['color']], marker='d',facecolor='none')

    axs[system_idx].set_title(f'{system} on Pt(111)')
    axs[system_idx].set_xlabel(r'$k$-point grid ($M \times M \times 1$)')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([0,15])
axs[0].set_xticks([0,2,4,6,8,10,12,14])
axs[1].legend()
fig.suptitle(r'$E_\text{ads}$ convergence with $k$-points')
axs[0].set_ylabel(r'$E_\text{ads}$ w.r.t. $14 \times 14 \times 1$ $k$-point grid (kJ/mol)')
plt.savefig('Figures/SI_Figure-Eads_KPOINTS_Convergence.png')

# Showcase effect of BEEF-vdW correction on the convergence of hBEEF-vdW and dhBEEF-vdW
fig, axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

delta_corr_hBEEFvdW = {system: {kpoint: (kpoint_conv_eads_dict[system][14]['BEEFvdW'] - kpoint_conv_eads_dict[system][kpoint]['BEEFvdW']) for kpoint in range(1,15)} for system in ['CO','H2O']}

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    # Plot hBEEFvdW and dhBEEFvdW with and without BEEF-vdW correction
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['hBEEFvdW'] - kpoint_conv_eads_dict[system][14]['hBEEFvdW']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='hBEEF-vdW', color=color_dict[method_dict['hBEEFvdW']['color']], marker='^',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['hBEEFvdW'] - kpoint_conv_eads_dict[system][14]['hBEEFvdW'] + delta_corr_hBEEFvdW[system][kpoint]) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label=r'hBEEF-vdW + $\Delta_k$', color='red', marker='^',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['dhBEEFvdW'] - kpoint_conv_eads_dict[system][14]['dhBEEFvdW']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='dhBEEF-vdW', color=color_dict[method_dict['dhBEEFvdW']['color']], marker='v',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['dhBEEFvdW'] - kpoint_conv_eads_dict[system][14]['dhBEEFvdW'] + delta_corr_hBEEFvdW[system][kpoint]) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label=r'dhBEEF-vdW + $\Delta_k$', color='red', marker='v',facecolor='none')
    axs[system_idx].set_title(f'{system} on Pt(111)')
    # RPA
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['RPA'] - kpoint_conv_eads_dict[system][14]['RPA']) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label='RPA', color=color_dict[method_dict['RPA']['color']], marker='s',facecolor='none')
    axs[system_idx].scatter([kpoint for kpoint in range(1,15)], [(kpoint_conv_eads_dict[system][kpoint]['RPA'] - kpoint_conv_eads_dict[system][14]['RPA'] + delta_corr_hBEEFvdW[system][kpoint]) * 1000 / kjmol_to_meV for kpoint in range(1,15)], label=r'RPA + $\Delta_k$', color='red', marker='s',facecolor='none')

    axs[system_idx].set_xlabel(r'$k$-point grid ($M \times M \times 1$)')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([0,15])
axs[0].set_xticks([0,2,4,6,8,10,12,14])
axs[1].legend(ncols=2,fontsize=8)
fig.suptitle(r'Effect of BEEF-vdW correction on $E_\text{ads}$ convergence with $k$-points')
axs[0].set_ylabel(r'$E_\text{ads}$ w.r.t. $14 \times 14 \times 1$ $k$-point grid (kJ/mol)')
plt.savefig('Figures/SI_Figure-Eads_KPOINTS_Convergence_BEEF_Correction.png')

In [96]:
encut_conv_eads_dict = {system: {encut: {method: 0 for method in ['EXX','EXX_w_0','EXX_w_1','EXX_w_2','EXX_w_3','RPAc','RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW'] } for encut in [150,200,250,300,310,350,400,450,500,550]} for system in ['CO', 'H2O']}

for encut in [150,200,250,300,310,350,400,450,500,550]:
    for system in ['CO', 'H2O']:
        encut_conv_eads_dict[system][encut]['EXX'] = get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/{system}_Pt/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/{system}/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/Pt/OUTCAR_02_EXX.gz', code='vasp')
        encut_conv_eads_dict[system][encut]['RPAc'] = get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/{system}_Pt/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/{system}/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/Pt/OUTCAR_04_RPA.gz', code='vasp_rpa')
        encut_conv_eads_dict[system][encut]['RPA'] = encut_conv_eads_dict[system][encut]['EXX'] + encut_conv_eads_dict[system][encut]['RPAc']
        encut_conv_eads_dict[system][encut]['BEEFvdW'] = get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/{system}_Pt/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/{system}/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT/{encut}/Pt/OUTCAR_01_DFT.gz', code='vasp')
        for w_idx in ['0','1','2','3']:
            encut_conv_eads_dict[system][encut][f'EXX_w_{w_idx}'] = get_energy(f'Data/Convergence/Eads/ENCUT_EXX/{encut}/{system}_Pt/OUTCAR_03_{w_idx}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT_EXX/{encut}/{system}/OUTCAR_03_{w_idx}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT_EXX/{encut}/Pt/OUTCAR_03_{w_idx}.gz', code='vasp')
        # hBEEF and dhBEEF
        calc_energy_dict = {
            '00': 0,
            '01': 0,
            '02': 0,
            '03_3': 0,
        }
        for calc_type in calc_energy_dict:
            calc_energy_dict[calc_type] = get_energy(f'Data/Convergence/Eads/ENCUT_EXX/{encut}/{system}_Pt/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT_EXX/{encut}/{system}/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/ENCUT_EXX/{encut}/Pt/OUTCAR_{calc_type}.gz', code='vasp')

        beefx_energy = calc_energy_dict['02']
        exx_energy = calc_energy_dict['03_3']
        nlc_energy = calc_energy_dict['00'] - calc_energy_dict['01']
        beefc_energy = calc_energy_dict['01'] - calc_energy_dict['02']
        rpac_energy = encut_conv_eads_dict[system][encut]['RPAc']
        dh_x_frac = 0.25
        dh_c_frac = 0.15
        encut_conv_eads_dict[system][encut]['dhBEEFvdW'] = dh_x_frac * exx_energy + (1-dh_x_frac)*beefx_energy + dh_c_frac*rpac_energy + (1-dh_c_frac)*beefc_energy + nlc_energy
        h_x_frac = 0.175
        encut_conv_eads_dict[system][encut]['hBEEFvdW'] = h_x_frac * exx_energy + (1-h_x_frac)*beefx_energy + beefc_energy + nlc_energy
        

# Plot for CO and H2O
fig,axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[system_idx].scatter([encut for encut in [150,200,250,300,310,350,400,450,500,550]], [(encut_conv_eads_dict[system][encut]['EXX'] - encut_conv_eads_dict[system][550]['EXX']) * 1000 / kjmol_to_meV for encut in [150,200,250,300,310,350,400,450,500,550]], label='EXX', color=color_dict['purple'], marker='x')
    # Plot convergence of hBEEFvdW and dhBEEFvdW
    axs[system_idx].scatter([encut for encut in [150,200,250,300,310,350,400,450,500,550]], [(encut_conv_eads_dict[system][encut]['hBEEFvdW'] - encut_conv_eads_dict[system][550]['hBEEFvdW']) * 1000 / kjmol_to_meV for encut in [150,200,250,300,310,350,400,450,500,550]], label='hBEEF-vdW', color=color_dict[method_dict['hBEEFvdW']['color']], marker='^',facecolor='none')
    axs[system_idx].scatter([encut for encut in [150,200,250,300,310,350,400,450,500,550]], [(encut_conv_eads_dict[system][encut]['dhBEEFvdW'] - encut_conv_eads_dict[system][550]['dhBEEFvdW']) * 1000 / kjmol_to_meV for encut in [150,200,250,300,310,350,400,450,500,550]], label='dhBEEF-vdW', color=color_dict[method_dict['dhBEEFvdW']['color']], marker='v',facecolor='none')

    axs[system_idx].scatter([encut for encut in [150,200,250,300,310,350,400,450,500,550]], [(encut_conv_eads_dict[system][encut]['RPAc'] - encut_conv_eads_dict[system][550]['RPAc']) * 1000 / kjmol_to_meV for encut in [150,200,250,300,310,350,400,450,500,550]], label='RPA correlation', color=color_dict['green'], marker='^',facecolor='none')
    axs[system_idx].scatter([encut for encut in [150,200,250,300,310,350,400,450,500,550]], [(encut_conv_eads_dict[system][encut]['RPA'] - encut_conv_eads_dict[system][550]['RPA']) * 1000 / kjmol_to_meV for encut in [150,200,250,300,310,350,400,450,500,550]], label='Total RPA', color=color_dict[method_dict['RPA']['color']], marker='s',facecolor='none')
    axs[system_idx].scatter([encut for encut in [150,200,250,300,310,350,400,450,500,550]], [(encut_conv_eads_dict[system][encut]['BEEFvdW'] - encut_conv_eads_dict[system][550]['BEEFvdW']) * 1000 / kjmol_to_meV for encut in [150,200,250,300,310,350,400,450,500,550]], label='BEEF-vdW', color=color_dict[method_dict['BEEFvdW']['color']], marker='d',facecolor='none')

    axs[system_idx].set_title(f'{system} on Pt(111)')
    axs[system_idx].set_xlabel(r'Energy cutoff (eV)')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([80,600])
axs[0].set_xticks([100,200,300,400,500,600])
axs[1].legend(ncols=2)
fig.suptitle(r'$E_\text{ads}$ convergence with plane-wave cutoff')
axs[0].set_ylabel(r'$E_\text{ads}$ w.r.t. 550 eV cutoff (kJ/mol)')
plt.savefig('Figures/SI_Figure-Eads_ENCUT_Convergence.png')

In [97]:
nomega_conv_eads_dict = {system: {nomega: {method: 0 for method in ['EXX','RPAc','RPA','BEEFvdW'] } for nomega in range(2,24,2)} for system in ['CO', 'H2O']}

for nomega in range(2,24,2):
    for system in ['CO', 'H2O']:
        nomega_conv_eads_dict[system][nomega]['EXX'] = get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/{system}_Pt/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/{system}/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/Pt/OUTCAR_02_EXX.gz', code='vasp')
        nomega_conv_eads_dict[system][nomega]['RPAc'] = get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/{system}_Pt/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/{system}/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/Pt/OUTCAR_04_RPA.gz', code='vasp_rpa')
        nomega_conv_eads_dict[system][nomega]['RPA'] = nomega_conv_eads_dict[system][nomega]['EXX'] + nomega_conv_eads_dict[system][nomega]['RPAc']
        nomega_conv_eads_dict[system][nomega]['BEEFvdW'] = get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/{system}_Pt/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/{system}/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/NOMEGA/{nomega}/Pt/OUTCAR_01_DFT.gz', code='vasp')

# Plot for CO and H2O
fig,axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[system_idx].scatter([nomega for nomega in range(2,24,2)], [(nomega_conv_eads_dict[system][nomega]['RPAc'] - nomega_conv_eads_dict[system][22]['RPAc']) * 1000 / kjmol_to_meV for nomega in range(2,24,2)], label='RPA correlation', color=color_dict['green'], marker='^',facecolor='none')

    axs[system_idx].set_title(f'{system} on Pt(111)')
    axs[system_idx].set_xlabel(r'Imaginary frequency grid points')
axs[0].set_ylim([-20,50])
axs[0].set_xlim([0,24])
axs[0].set_xticks([0,4,8,12,16,20,24])
axs[1].legend(ncols=2)
fig.suptitle(r'$E_\text{ads}$ convergence with imaginary frequency points')
axs[0].set_ylabel(r'$E_\text{ads}$ w.r.t. 20 grid points (kJ/mol)')
plt.savefig('Figures/SI_Figure-Eads_NOMEGA_Convergence.png')

In [98]:
precfock_conv_eads_dict = {system: {precfock: {method: 0 for method in ['EXX','RPAc','RPA','BEEFvdW'] } for precfock in ['Fast','Normal', 'Accurate']} for system in ['CO', 'H2O']}

for precfock in ['Fast','Normal', 'Accurate']:
    for system in ['CO', 'H2O']:
        precfock_conv_eads_dict[system][precfock]['EXX'] = get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/{system}_Pt/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/{system}/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/Pt/OUTCAR_02_EXX.gz', code='vasp')
        precfock_conv_eads_dict[system][precfock]['RPAc'] = get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/{system}_Pt/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/{system}/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/Pt/OUTCAR_04_RPA.gz', code='vasp_rpa')
        precfock_conv_eads_dict[system][precfock]['RPA'] = precfock_conv_eads_dict[system][precfock]['EXX'] + precfock_conv_eads_dict[system][precfock]['RPAc']
        precfock_conv_eads_dict[system][precfock]['BEEFvdW'] = get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/{system}_Pt/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/{system}/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/PRECFOCK/{precfock}/Pt/OUTCAR_01_DFT.gz', code='vasp')
# Plot for CO and H2O
fig,axs = plt.subplots(1,2, figsize=(6.65,3), dpi=450, constrained_layout=True, sharey='row', sharex=True)

for system_idx, system in enumerate(['CO','H2O']):
    axs[system_idx].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axs[system_idx].scatter(['Fast','Normal','Accurate'], [(precfock_conv_eads_dict[system][precfock]['EXX'] - precfock_conv_eads_dict[system]['Accurate']['EXX']) * 1000 / kjmol_to_meV for precfock in ['Fast','Normal','Accurate']], label='EXX', color=color_dict['purple'], marker='x')
    axs[system_idx].scatter(['Fast','Normal','Accurate'], [(precfock_conv_eads_dict[system][precfock]['RPAc'] - precfock_conv_eads_dict[system]['Accurate']['RPAc']) * 1000 / kjmol_to_meV for precfock in ['Fast','Normal','Accurate']], label='RPA correlation', color=color_dict['green'], marker='^',facecolor='none')
    axs[system_idx].scatter(['Fast','Normal','Accurate'], [(precfock_conv_eads_dict[system][precfock]['RPA'] - precfock_conv_eads_dict[system]['Accurate']['RPA']) * 1000 / kjmol_to_meV for precfock in ['Fast','Normal','Accurate']], label='Total RPA', color=method_dict['RPA']['color'], marker='s',facecolor='none')
    axs[system_idx].scatter(['Fast','Normal','Accurate'], [(precfock_conv_eads_dict[system][precfock]['BEEFvdW'] - precfock_conv_eads_dict[system]['Accurate']['BEEFvdW']) * 1000 / kjmol_to_meV for precfock in ['Fast','Normal','Accurate']], label='BEEF-vdW', color=method_dict['BEEFvdW']['color'], marker='d',facecolor='none')

    axs[system_idx].set_title(f'{system} on Pt(111)')
    axs[system_idx].set_xlabel('PRECFOCK setting')
axs[0].set_ylim([-20,20])
axs[0].set_xlim([-0.5,2.5])
axs[0].set_xticks([0,1,2])
axs[0].set_xticklabels(['Fast','Normal','Accurate'])
axs[1].legend(ncols=2)
fig.suptitle(r'$E_\text{ads}$ convergence with PRECFOCK setting')
axs[0].set_ylabel(r'$E_\text{ads}$ w.r.t. Accurate PRECFOCK (kJ/mol)')
plt.savefig('Figures/SI_Figure-Eads_PRECFOCK_Convergence.png')

In [100]:
kpoints_nsc_conv_eads_dict = {system: {kpoint: {method: 0 for method in ['RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW']} for kpoint in range(1,15)} for system in ['CO', 'H2O']}

for kpoint in range(1,15):
    for system in ['CO', 'H2O']:
        # Compute the RPA total energy first
        exx_eads = get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}_Pt/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/Pt/OUTCAR_02_EXX.gz', code='vasp')
        rpac_eads = get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}_Pt/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/Pt/OUTCAR_04_RPA.gz', code='vasp_rpa')
        kpoints_nsc_conv_eads_dict[system][kpoint]['RPA'] = exx_eads + rpac_eads
        # Then BEEF-vdW
        kpoints_nsc_conv_eads_dict[system][kpoint]['BEEFvdW'] = get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}_Pt/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/{system}/OUTCAR_01_DFT.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS/{kpoint}/Pt/OUTCAR_01_DFT.gz', code='vasp')
        # Then hBEEF-vdW
        calc_energy_dict = {
            '00': 0,
            '01': 0,
            '02': 0,
            '03_3': 0,
        }
        # hBEEF part
        for calc_type in calc_energy_dict:
            calc_energy_dict[calc_type] = get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}_Pt/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/{system}/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'Data/Convergence/Eads/KPOINTS_EXX/{kpoint}/Pt/OUTCAR_{calc_type}.gz', code='vasp')

        beefx_eads = calc_energy_dict['02']
        exx_eads = calc_energy_dict['03_3']
        nlc_eads = calc_energy_dict['00'] - calc_energy_dict['01']
        beefc_eads = calc_energy_dict['01'] - calc_energy_dict['02']
        dh_x_frac = 0.25
        dh_c_frac = 0.15
        kpoints_nsc_conv_eads_dict[system][kpoint]['dhBEEFvdW'] = dh_x_frac * exx_eads + (1-dh_x_frac)*beefx_eads + dh_c_frac*rpac_eads + (1-dh_c_frac)*beefc_eads + nlc_eads
        h_x_frac = 0.175
        kpoints_nsc_conv_eads_dict[system][kpoint]['hBEEFvdW'] = h_x_frac * exx_eads + (1-h_x_frac)*beefx_eads + beefc_eads + nlc_eads



# CE39 dataset

## Experimental data

In [101]:
raw_data = { # Taken from Surf. Sci. 640, 36–44 (2015).
    "#": list(range(1, 40)),
    "Surface reaction": [
        "CO + Ni(111) → CO/Ni(111)",
        "CO + Pt(111) → CO/Pt(111)",
        "CO + Pd(111) → CO/Pd(111)",
        "CO + Pd(100) → CO/Pd(100)",
        "CO + Rh(111) → CO/Rh(111)",
        "CO + Ir(111) → CO/Ir(111)",
        "CO + Cu(111) → CO/Cu(111)",
        "CO + Ru(001) → CO/Ru(001)",
        "CO + Co(001) → CO/Co(001)",
        "NO + Ni(100) → N/Ni(100) + O/Ni(100)",
        "NO + Pt(111) → NO/Pt(111)",
        "NO + Pd(111) → NO/Pd(111)",
        "NO + Pd(100) → NO/Pd(100)",
        "O2 + Ni(111) → 2O/Ni(111)",
        "O2 + Ni(100) → 2O/Ni(100)",
        "O2 + Pt(111) → 2O/Pt(111)",
        "O2 + Rh(100) → 2O/Rh(100)",
        "H2 + Pt(111) → 2H/Pt(111)",
        "H2 + Ni(111) → 2H/Ni(111)",
        "H2 + Ni(100) → 2H/Ni(100)",
        "H2 + Rh(111) → 2H/Rh(111)",
        "H2 + Pd(111) → 2H/Pd(111)",
        "I2 + Pt(111) → 2I/Pt(111)",
        "CH2I2 + Pt(111) → CH/Pt(111) + H/Pt(111) + 2I/Pt(111)",
        "CH3I + Pt(111) → CH3/Pt(111) + I/Pt(111)",
        "NH3 + Cu(100) → NH3/Cu(100)",
        "CH3I + Pt(111) → CH3I/Pt(111)",
        "CH3OH + Pt(111) → CH3OH/Pt(111)",
        "CH4 + Pt(111) → CH4/Pt(111)",
        "C2H6 + Pt(111) → C2H6/Pt(111)",
        "C3H8 + Pt(111) → C3H8/Pt(111)",
        "C4H10 + Pt(111) → C4H10/Pt(111)",
        "C6H6 + Pt(111) → C6H6/Pt(111)",
        "C6H6 + Cu(111) → C6H6/Cu(111)",
        "C6H6 + Ag(111) → C6H6/Ag(111)",
        "C6H6 + Au(111) → C6H6/Au(111)",
        "C6H10 + Pt(111) → C6H10/Pt(111)",
        "H2O + Pt(111) → H2O/Pt(111)",
        "H2O + OPt(111) → H2OOH/Pt(111)"
    ],
    "Coverage (ML)": [
        "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/8", "1/4", "1/4", "1/4", "1/8",
        "1/8", "1/18", "1/8", "1/8", "1/8", "1/8", "1/8", "1/8", "1/4", "1/4", "1/4", "1/4", "1/4",
        "1/4", "1/4", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/4", "1/2"
    ],
    "Reaction enthalpy (kJ/mol)": [
        -122, -120, -143, -155, -139, -158, -53, -158, -115, -290, -114, -179, -161, -480,
        -530, -208, -358, -72, -94, -94, -70, -88, -312, -473, -212, -57, -84.5, -56, -15,
        -28.5, -41.3, -50.8, -164, -68, -63, -72, -122, -51.3, -57.4
    ],
    "Temp. (K)": [
        350, 340, 450, 430, 500, 420, 350, 475, 370, 300, 300, 520, 300, 300,
        300, 515, 300, 300, 370, 370, 325, 370, 0, 210, 320, 235, 100, 100,
        63, 106, 139, 171, 300, 225, 210, 230, 100, 120, 150
    ],
    "Reaction energy (kJ/mol)": [
        -119, -117, -139, -151, -135, -154, -52, -154, -112, -288, -112, -175, -159, -479,
        -528, -204, -356, -70, -91, -91, -67, -85, -230, -471, -209, -55, -83.7, -55,
        -14.5, -27.6, -40.1, -49.4, -300, -66, -61, -70, -121, -50.3, -56.2
    ],
    "Reaction energy without ZPE (kJ/mol)": [
        -124, -124, -144, -157, -142, -164, -57, -161, -119, -299,
        -119, -182, -163, -485, -530, -208, -355, -72, -100, -87,
        -72, -90, -313, -455, -209
    ] + [
        -60, -84, -55, -14, -27, -39, -48, -162, -66, -61,
        -70, -123, -55, -66
    ],
    "Weight": [ 1, 1, 1, 1, 1, 1, 1, 1, 1,0.5, 1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,0.5, 0.5, 0.5, 0.5, 0.25, 0.5] + [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,1]
}

ce39_dataset_dict = {}

calc_molecule_list = []
calc_molecule_surface_list = []

for ce39_idx in range(39):


    if 'Co' in raw_data['Surface reaction'][ce39_idx] or 'Ru' in raw_data['Surface reaction'][ce39_idx]:
        crystal_type = 'hcp'
    else:
        crystal_type = 'fcc'

    # Set the supercell_size based on coverage
    if raw_data['Coverage (ML)'][ce39_idx] in ['1/4','1/8']:
        surface_sc_size = '2x2'
    else:
        surface_sc_size = '3x3'

    surface_type = raw_data['Surface reaction'][ce39_idx].split()[2].replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')
    system_molecule = raw_data['Surface reaction'][ce39_idx].split()[0]
    molecule_surface_system_list = []
    for molecule_surface_system in raw_data['Surface reaction'][ce39_idx].split('→')[1].split('+'):
        if molecule_surface_system.strip()[0].isdigit():
            molecule_surface_system_list += [molecule_surface_system.strip()[1:].replace('/','-').replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')]*int(molecule_surface_system.strip()[0])
        else:
            molecule_surface_system_list += [molecule_surface_system.strip().replace('/','-').replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')]


    ce39_dataset_dict[f'{ce39_idx+1:02d}'] = {'Reaction': raw_data['Surface reaction'][ce39_idx], 'MAD Weight': raw_data['Weight'][ce39_idx], 'Reactant Molecule': [system_molecule], 'Reactant Surface': [surface_type], 'Products': molecule_surface_system_list, 'Reaction Enthalpy': int(raw_data["Reaction enthalpy (kJ/mol)"][ce39_idx]), 'Reaction Energy': int(raw_data["Reaction energy (kJ/mol)"][ce39_idx]),'Reaction Energy without ZPE': int(raw_data["Reaction energy without ZPE (kJ/mol)"][ce39_idx])}

    for molecule_surface_system in molecule_surface_system_list:
        if molecule_surface_system not in calc_molecule_surface_list:
            calc_molecule_surface_list += [molecule_surface_system]
    
    if system_molecule not in calc_molecule_list:
        calc_molecule_list += [system_molecule]

ce39_dataset_df = pd.DataFrame.from_dict(ce39_dataset_dict, orient='index')
display(ce39_dataset_df)

,Reaction,MAD Weight,Reactant Molecule,Reactant Surface,Products,Reaction Enthalpy,Reaction Energy,Reaction Energy without ZPE
01,CO + Ni(111) → CO/Ni(111),1.00,[CO],[Ni_fcc_111_2x2],[CO-Ni_fcc_111_2x2],-122,-119,-124
02,CO + Pt(111) → CO/Pt(111),1.00,[CO],[Pt_fcc_111_2x2],[CO-Pt_fcc_111_2x2],-120,-117,-124
03,CO + Pd(111) → CO/Pd(111),1.00,[CO],[Pd_fcc_111_2x2],[CO-Pd_fcc_111_2x2],-143,-139,-144
04,CO + Pd(100) → CO/Pd(100),1.00,[CO],[Pd_fcc_100_2x2],[CO-Pd_fcc_100_2x2],-155,-151,-157
05,CO + Rh(111) → CO/Rh(111),1.00,[CO],[Rh_fcc_111_2x2],[CO-Rh_fcc_111_2x2],-139,-135,-142
06,CO + Ir(111) → CO/Ir(111),1.00,[CO],[Ir_fcc_111_2x2],[CO-Ir_fcc_111_2x2],-158,-154,-164
07,CO + Cu(111) → CO/Cu(111),1.00,[CO],[Cu_fcc_111_2x2],[CO-Cu_fcc_111_2x2],-53,-52,-57
08,CO + Ru(001) → CO/Ru(001),1.00,[CO],[Ru_hcp_001_2x2],[CO-Ru_hcp_001_2x2],-158,-154,-161
09,CO + Co(001) → CO/Co(001),1.00,[CO],[Co_hcp_001_2x2],[CO-Co_hcp_001_2x2],-115,-112,-119
10,NO + Ni(100) → N/Ni(100) + O/Ni(100),0.50,[NO],[Ni_fcc_100_2x2],"[N-Ni_fcc_100_2x2, O-Ni_fcc_100_2x2]",-290,-288,-299


## DFT estimates

### BEEF-vdW

In [102]:
# Calculate the adsorption energy from BEEF-vdW with a 4L and 6L slab
ce39_eads_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0, 'BEEFvdW (4L)': 0, 'BEEFvdW (6L)': 0} for ce39_idx in ce39_dataset_dict}

for method in ['4L','6L']:
    if method == '4L':
        root_folder = f'Data/CE39/Eads_BEEFvdW_4L'
    elif method == '6L':
        root_folder = f'Data/CE39/Eads_BEEFvdW_6L'

    for ce39_idx in ce39_dataset_dict:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)

        total_product_energy = 0
        for product_idx in range(num_products):
            total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/{method}_{products[product_idx]}/OUTCAR.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = get_energy(f'{root_folder}/Surface/{method}_{reactant_surface}/OUTCAR.gz', code='vasp')
            ce39_eads_dict[ce39_idx][f'BEEFvdW ({method})'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
        else:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/{method}_O-Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/{method}_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') 
            ce39_eads_dict[ce39_idx][f'BEEFvdW ({method})'] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV

        ce39_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
        ce39_eads_dict[ce39_idx]['Reaction'] = ce39_dataset_dict[ce39_idx]['Reaction']

# Convert to DataFrame
ce39_eads_df = pd.DataFrame.from_dict(ce39_eads_dict, orient='index')
mad_dict = {method: np.mean(abs(ce39_eads_df[method] - ce39_eads_df['Experiment']) * ce39_dataset_df['MAD Weight']) for method in ['BEEFvdW (4L)', 'BEEFvdW (6L)']}
# Add an additional row at the bottom that is the MAD for each method
ce39_eads_df.loc['MAD'] = ['MAD', 0, mad_dict['BEEFvdW (4L)'], mad_dict['BEEFvdW (6L)']]

# Round to nearest integer and convert to integer
ce39_eads_df = ce39_eads_df.round(0).astype({'Experiment': 'Int64', 'BEEFvdW (4L)': 'Int64', 'BEEFvdW (6L)': 'Int64', 'Reaction': str}) 

display(ce39_eads_df)

,Reaction,Experiment,BEEFvdW (4L),BEEFvdW (6L)
01,CO + Ni(111) → CO/Ni(111),-124,-150,-151
02,CO + Pt(111) → CO/Pt(111),-124,-133,-135
03,CO + Pd(111) → CO/Pd(111),-144,-158,-148
04,CO + Pd(100) → CO/Pd(100),-157,-152,-150
05,CO + Rh(111) → CO/Rh(111),-142,-161,-163
06,CO + Ir(111) → CO/Ir(111),-164,-172,-179
07,CO + Cu(111) → CO/Cu(111),-57,-52,-50
08,CO + Ru(001) → CO/Ru(001),-161,-162,-156
09,CO + Co(001) → CO/Co(001),-119,-135,-137
10,NO + Ni(100) → N/Ni(100) + O/Ni(100),-299,-393,-385


## Comparison between NSC-DFAs, RPA and DFT

In [103]:
# Analyse the interaction energy calculations
ce39_int_dict = {ce39_idx: {'RPA': 0 , 'BEEFvdW': 0, 'hBEEFvdW': 0,'dhBEEFvdW': 0} for ce39_idx in ce39_dataset_dict}
final_ce39_eads_dict = {
    ce39_idx:
        {
            'Reaction': '',
            'Experiment': 0,
            'RPA': 0,
            'BEEFvdW': 0,
            'hBEEFvdW': 0,
            'dhBEEFvdW': 0,
        }
        | {dft_xc: 0 for dft_xc in dft_xc_list}
    for ce39_idx in ce39_dataset_dict
}

# Start by analyzing BEEFvdW
root_folder = "Data/CE39/Eint_BEEFvdW"
for ce39_idx in ce39_int_dict:
    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)

    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR.gz', code='vasp')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR.gz', code='vasp')
        reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR.gz', code='vasp')
        ce39_int_dict[ce39_idx]['BEEFvdW'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp')
        reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') 
        ce39_int_dict[ce39_idx]['BEEFvdW'] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV


# Analyze hBEEFvdW, dhBEEFvdW and RPA
root_folder = f'Data/CE39/Eint_hBEEFvdW'
rpa_root_folder = f'Data/CE39/Eint_RPA'

for ce39_idx in ce39_int_dict:
    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)
    calc_energy_dict = {
        '00': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '01': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '02': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '03_3': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0}
    }
    # hBEEF part
    for calc_type in calc_energy_dict:
        for product_idx in range(num_products):
            calc_energy_dict[calc_type]['total_product_energy'] += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{calc_type}.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['reactant_surface_energy'] = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['eint'] = (calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - num_products * calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV
        else:
            calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['reactant_surface_energy'] = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp') - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp') 
            calc_energy_dict[calc_type]['eint'] = (2 / 9 * calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV

    # RPA correlation part
    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz', code='vasp_rpa')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        rpac_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        reactant_surface_energy = (1/3)*get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa')- (1/9)*get_energy(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa') 
        rpac_eint = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV

    # RPA EXX part
    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_02_EXX.gz', code='vasp')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_02_EXX.gz', code='vasp')
        reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_02_EXX.gz', code='vasp')
        rpax_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp')
        reactant_surface_energy = (1/3)*get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp') - (1/9)*get_energy(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp') 
        rpax_eint = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV


    beefx_eint = calc_energy_dict['02']['eint']
    exx_eint = calc_energy_dict['03_3']['eint']
    nlc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['01']['eint']
    beefc_eint = calc_energy_dict['01']['eint'] - calc_energy_dict['02']['eint']
    dh_x_frac = 0.25
    dh_c_frac = 0.15
    ce39_int_dict[ce39_idx]['dhBEEFvdW'] = dh_x_frac * exx_eint + (1-dh_x_frac)*beefx_eint + dh_c_frac*rpac_eint + (1-dh_c_frac)*beefc_eint + nlc_eint
    h_x_frac = 0.175
    ce39_int_dict[ce39_idx]['hBEEFvdW'] = h_x_frac * exx_eint + (1-h_x_frac)*beefx_eint + beefc_eint + nlc_eint
    ce39_int_dict[ce39_idx]['RPA'] = rpax_eint + rpac_eint

    final_ce39_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
    final_ce39_eads_dict[ce39_idx]['Reaction'] = r'\ce{' + ce39_dataset_dict[ce39_idx]['Reaction'] + r'}'


# Convert interaction energy to adsorption energy

for method in ['RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW']:
    for ce39_idx in ce39_int_dict:
        final_ce39_eads_dict[ce39_idx][method] = ce39_int_dict[ce39_idx][method] - ce39_int_dict[ce39_idx]['BEEFvdW'] + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)']

# Calculate the adsorption energy for each DFT XC functional and convert to adsorption energy using the interaction energy from BEEF-vdW
for dft_xc in dft_xc_list:
    root_folder = f'Data/CE39/Eads_DFT_4L'
    for ce39_idx in final_ce39_eads_dict:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)

        total_product_energy = 0
        for product_idx in range(num_products):
            total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{dft_xc}.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}.gz', code='vasp')
            reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{dft_xc}.gz', code='vasp')
            final_ce39_eads_dict[ce39_idx][dft_xc] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']
        else:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}.gz', code='vasp')
            reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}.gz', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}.gz', code='vasp') 
            final_ce39_eads_dict[ce39_idx][dft_xc] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']


In [104]:
metrics_methods = ['RPA', 'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW'] + dft_xc_list

chemisorption_list = [f'{ce39_idx:02d}' for ce39_idx in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]]
physisorption_list = [f'{ce39_idx:02d}' for ce39_idx in [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]]

# Combine final_ce39_eads_dict and ce39_dft_xc_eads_dict into a single DataFrame
final_ce39_eads_df = pd.DataFrame.from_dict(final_ce39_eads_dict, orient='index')

# MAD
total_mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df[m] - final_ce39_eads_df['Experiment']) * ce39_dataset_df['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['T'] = ['MAD (Total)', 0] + [total_mad_dict_final[m] for m in metrics_methods]

# MAD Physisorption
physisorption_mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df.loc[physisorption_list][m] - final_ce39_eads_df.loc[physisorption_list]['Experiment']) * ce39_dataset_df.loc[physisorption_list]['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['P'] = ['MAD (Physisorption)', 0] + [physisorption_mad_dict_final[m] for m in metrics_methods]

# MAD Chemisorption
chemisorption_mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df.loc[chemisorption_list][m] - final_ce39_eads_df.loc[chemisorption_list]['Experiment']) * ce39_dataset_df.loc[chemisorption_list]['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['C'] = ['MAD (Chemisorption)', 0] + [chemisorption_mad_dict_final[m] for m in metrics_methods]

# Round and cast
astype_map = {'Experiment': 'Int64', 'Reaction': str}
astype_map.update({m: 'Int64' for m in metrics_methods})
final_ce39_eads_df = final_ce39_eads_df.round(0).astype(astype_map)

# Update column order to be 'Reaction', 'Experiment', 'dhBEEFvdW', 'hBEEFvdW', 'RPA', 'BEEFvdW' followed by the DFT XC functionals in the order of dft_xc_list
final_ce39_eads_df = final_ce39_eads_df[['Reaction', 'Experiment', 'dhBEEFvdW', 'hBEEFvdW', 'RPA', 'BEEFvdW'] + dft_xc_list]
# Change the column names to be more readable
final_ce39_eads_df.columns = ['Reaction', 'Experiment', 'dhBEEF-vdW@BEEF-vdW', 'hBEEF-vdW@BEEF-vdW', 'RPA@PBE', 'BEEF-vdW'] + [method_dict[dft_xc]['label'] for dft_xc in dft_xc_list]

# Make a shortened final_ce39_eads_df with just ['Reaction', 'Experiment', 'dhBEEF-vdW@BEEF-vdW', 'hBEEF-vdW@BEEF-vdW', 'RPA@PBE', 'BEEF-vdW']
final_ce39_eads_df_nsc = final_ce39_eads_df[['Reaction', 'Experiment', 'dhBEEF-vdW@BEEF-vdW', 'hBEEF-vdW@BEEF-vdW', 'RPA@PBE', 'BEEF-vdW']]
# Make another shortened final_ce39_eads_df with just ['Reaction', 'Experiment'] + dft_xc_list[:8]
final_ce39_eads_df_dft = final_ce39_eads_df[['Experiment'] + [method_dict[dft_xc]['label'] for dft_xc in dft_xc_list]]

display(final_ce39_eads_df)

convert_df_to_latex_input(
    df = final_ce39_eads_df_nsc,
    start_input = '\\begin{table}[h]\n',
    label = "tab:ce39_eads_1",
    caption = r"Adsorption energies for the CE39 dataset calculated with dhBEEF-vdW@BEEF-vdW, hBEEF-vdW@BEEF-vdW, RPA@PBE, BEEF-vdW compared against experiment. All values are in kJ/mol.",
    replace_input= {
        "[t]": "[c]",
        r"→" : r"$\rightarrow$",
        r" & \rotatebox{90}{Reaction}": r"\rotatebox{90}{Index} & Reaction",
        r" OPt(111)": r" O/Pt(111)"
    },
    index = True,
    rotate_column_header=True,
    longtable=True,
    float_fmt="%.0f",
    filename = "Tables/SI_Table-CE39_Adsorption_Energies_NSC.tex"
)

convert_df_to_latex_input(
    df = final_ce39_eads_df_dft,
    start_input = '\\begin{table}[h]\n',
    label = "tab:ce39_eads_2",
    caption = r"Adsorption energies for the CE39 dataset calculated with several density functional approximation compared against experiment. All values are in kJ/mol.",
    replace_input= {
        "[t]": "[c]",
        r"→" : r"$\rightarrow$",
        r" & \rotatebox{90}{Experiment}": r"\rotatebox{90}{Index}  & \rotatebox{90}{Experiment}"
    },
    index = True,
    rotate_column_header=True,
    longtable=True,
    float_fmt="%.0f",
    filename = "Tables/SI_Table-CE39_Adsorption_Energies_DFA.tex"
)


,Reaction,Experiment,dhBEEF-vdW@BEEF-vdW,hBEEF-vdW@BEEF-vdW,RPA@PBE,BEEF-vdW,PBE,PBEsol,RPBE,revPBE,BLYP,r$^2$SCAN,MS2,revTPSS,PBE-D3,PBE-DdsC,RPBE-D3,optPBE-vdW,rev-vdW-DF2,r$^2$SCAN+rVV10
01,\ce{CO + Ni(111) → CO/Ni(111)},-124,-140,-139,-100,-151,-220,-229,-143,-187,-126,-186,-148,-174,-200,-200,-206,-179,-200,-202
02,\ce{CO + Pt(111) → CO/Pt(111)},-124,-147,-155,-135,-135,-165,-205,-133,-137,-110,-182,-176,-164,-197,-183,-213,-160,-180,-198
03,\ce{CO + Pd(111) → CO/Pd(111)},-144,-149,-164,-136,-148,-182,-229,-143,-148,-123,-191,-172,-174,-202,-197,-213,-176,-199,-207
04,\ce{CO + Pd(100) → CO/Pd(100)},-157,-153,-162,-144,-150,-180,-220,-146,-150,-132,-192,-179,-170,-202,-194,-208,-176,-193,-206
05,\ce{CO + Rh(111) → CO/Rh(111)},-142,-170,-178,-140,-163,-185,-215,-157,-160,-145,-197,-182,-178,-204,-202,-206,-184,-196,-211
06,\ce{CO + Ir(111) → CO/Ir(111)},-164,-188,-196,-159,-179,-198,-228,-171,-174,-158,-213,-189,-190,-220,-212,-222,-198,-209,-227
07,\ce{CO + Cu(111) → CO/Cu(111)},-57,-52,-44,-44,-50,-70,-99,-44,-47,-33,-82,-69,-70,-92,-87,-109,-72,-82,-96
08,\ce{CO + Ru(001) → CO/Ru(001)},-161,-162,-167,-142,-156,-177,-208,-150,-153,-134,-186,-175,-173,-199,-189,-213,-176,-189,-201
09,\ce{CO + Co(001) → CO/Co(001)},-119,-134,-120,-106,-137,-161,-197,-130,-134,-118,-161,-188,-159,-182,-178,-203,-163,-178,-176
10,\ce{NO + Ni(100) → N/Ni(100) + O/Ni(100)},-299,-393,-405,-407,-385,-505,-508,-369,-451,-382,-491,-535,-450,-453,-445,-442,-449,-481,-511


## Computational Cost

In [105]:
# Get computational cost data
ce39_cost_dict = {ce39_idx: {'BEEF-vdW': 0, 'EXX':0, 'RPA':0} for ce39_idx in ce39_dataset_dict}
ce39_ratio_dict = {ce39_idx: {'BEEF-vdW': 1, 'EXX':0, 'RPA':0} for ce39_idx in ce39_dataset_dict}

for ce39_idx in ce39_cost_dict:
    exx_root_folder = f'Data/CE39/Eint_hBEEFvdW'
    rpa_root_folder = f'Data/CE39/Eint_RPA'
    beef_root_folder = f'Data/CE39/Eint_hBEEFvdW'

    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)

    # BEEF-vdW cost
    total_cost = 0
    for product_idx in range(num_products):
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_00.gz')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_00.gz')
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_00.gz')
    else:
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_00.gz')
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_00.gz') + get_vasp_walltime(f'{beef_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_00.gz')
    ce39_cost_dict[ce39_idx]['BEEF-vdW'] = (total_cost/3600)*192

    # EXX cost
    total_cost = 0
    for product_idx in range(num_products):
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_03_3.gz')
    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_03_3.gz')
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_03_3.gz')
    else:
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_03_3.gz')
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_03_3.gz') + get_vasp_walltime(f'{exx_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_03_3.gz')
    ce39_cost_dict[ce39_idx]['EXX'] = (total_cost/3600)*192

    # RPA cost
    total_cost = 0
    for product_idx in range(num_products):
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz')
    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz')
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz')
    else:
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz')
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz') + get_vasp_walltime(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz')
    ce39_cost_dict[ce39_idx]['RPA'] = (total_cost/3600)*192
    
for ce39_idx in ce39_ratio_dict:
    ce39_ratio_dict[ce39_idx]['EXX'] = ce39_cost_dict[ce39_idx]['EXX'] / ce39_cost_dict[ce39_idx]['BEEF-vdW']
    ce39_ratio_dict[ce39_idx]['RPA'] = ce39_cost_dict[ce39_idx]['RPA'] / ce39_cost_dict[ce39_idx]['BEEF-vdW']

# Convert to DataFrame and display
ce39_ratio_df = pd.DataFrame.from_dict(ce39_ratio_dict, orient='index')
# Add name to the index
ce39_ratio_df.index.name = 'Reaction'
# Add an additional row at the bottom that is the mean ratio for each method
ce39_ratio_df.index = ce39_ratio_df.index.map(lambda x: r'\ce{' + ce39_dataset_dict[x]['Reaction'] + r'}')
ce39_ratio_df.loc['Mean'] = ce39_ratio_df.mean()
# Round to nearest integer
ce39_ratio_df = ce39_ratio_df.round(0).astype(int)
# Change the index columns to be the reaction string instead of the index

display(ce39_ratio_df)

convert_df_to_latex_input(
    df=ce39_ratio_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:ce39_cost_ratio",
    caption = r"Ratio between the computational cost of EXX and RPA calculations compared to BEEF-vdW for the CE39 dataset.",
    longtable=True,
    replace_input= {
        r'→': r'$\rightarrow$',
        r' OPt(111)': r' O/Pt(111)'
    },
    filename = "Tables/SI_Table-CE39_Cost_Ratio.tex")

,BEEF-vdW,EXX,RPA
Reaction,,,
\ce{CO + Ni(111) → CO/Ni(111)},1,18,24
\ce{CO + Pt(111) → CO/Pt(111)},1,23,36
\ce{CO + Pd(111) → CO/Pd(111)},1,25,45
\ce{CO + Pd(100) → CO/Pd(100)},1,27,56
\ce{CO + Rh(111) → CO/Rh(111)},1,18,52
\ce{CO + Ir(111) → CO/Ir(111)},1,15,29
\ce{CO + Cu(111) → CO/Cu(111)},1,22,35
\ce{CO + Ru(001) → CO/Ru(001)},1,14,73
\ce{CO + Co(001) → CO/Co(001)},1,18,24


In [106]:
# Updated 7-method list
methods_to_plot = ['01_PBE','03_RPBE','07_MS2','12_optPBEvdW','14_r2SCANrVV10',
                   'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW','RPA']

# 7 colors
colors = ['blue', 'orange', 'green', 'red', 'purple', 'cyan', 'magenta','black', 'gray']

fig, axs = plt.subplots(4, 2, figsize=(6.559,5.9), dpi=600, sharex='col', constrained_layout=True, height_ratios=[1,0.5,1,3.5])

bar_width = 0.6
x = np.arange(len(methods_to_plot))

# grab the GridSpec
gs = axs[0,0].get_gridspec()

# remove the two axes in the first column
axs[0,0].remove()
axs[1,0].remove()

# create one big axis spanning rows 0–1, column 0
ax_chem = fig.add_subplot(gs[:2, 0])

# plot wmad_dict_chem
ax_chem.bar(
    x,
    [chemisorption_mad_dict_final[m] for m in methods_to_plot],
    color=[color_dict[method_dict[m]['color']] for m in methods_to_plot],
    width=bar_width
)

ax_chem.set_title("Chemisorption", fontsize=10)


for i, v in enumerate([chemisorption_mad_dict_final[m] for m in methods_to_plot]):
    ax_chem.text(i, v + 2, f'{v:.1f}',
                 ha='center', fontsize=8, fontweight='bold')
    
ax_chem.set_xticklabels([])

########################################
# PANEL 3 — PHYSISORPTION
########################################
axs[2,0].bar(x, [physisorption_mad_dict_final[m] for m in methods_to_plot],
           color=[color_dict[method_dict[m]['color']] for m in methods_to_plot], width=bar_width)
axs[2,0].set_title("Physisorption", fontsize=10)

for i, v in enumerate([physisorption_mad_dict_final[m] for m in methods_to_plot]):
    axs[2,0].text(i, v + 2, f'{v:.1f}', ha='center', fontsize=8, fontweight='bold')

########################################
# PANEL 4 — TOTAL CE39 (now LAST)
########################################
for m_idx, m in enumerate(methods_to_plot):
    axs[3,0].bar(x[m_idx], total_mad_dict_final[m],
               color=color_dict[method_dict[m]['color']], width=bar_width, label = method_dict[m]['label'])


# axs[3,0].bar(x, [wmad_dict_final[m] for m in methods_to_plot],
#            color=colors, width=bar_width)
axs[3,0].set_title("Total", fontsize=10)

for i, v in enumerate([total_mad_dict_final[m] for m in methods_to_plot]):
    axs[3,0].text(i, v + 1, f'{v:.1f}', ha='center', fontsize=8, fontweight='bold')

########################################
# Shared X-axis
########################################
axs[3,0].set_xticks(x)
for i, v in enumerate([m for m in methods_to_plot]):
    axs[3,0].text(i+0.03, 1, method_dict[v]['label'], ha='center', fontsize=8, fontweight='bold',rotation = 90)

# Add shaded square from 0 to 12 on both axes
axs[3,1].fill_between([0, 20], 0, 20, color=color_dict['yellow'], alpha=0.6, edgecolor = 'none', label = '< 20 kJ/mol')
axs[3,1].fill_between([0, 13], 0, 13, color=color_dict['grey'], alpha=0.6, edgecolor = 'none', label = '< 13 kJ/mol')

chemisorption_mad_list = np.array([chemisorption_mad_dict_final[m] for m in method_dict])
physisorption_mad_list = np.array([physisorption_mad_dict_final[m] for m in method_dict])


axs[3,1].scatter(chemisorption_mad_list, physisorption_mad_list, edgecolors='k', facecolors=[color_dict[method_dict[m]['color']] for m in method_dict], s=30)
for method in method_dict:
    if method == 'dhBEEFvdW':
        axs[3,1].text(chemisorption_mad_dict_final[method] - 13, physisorption_mad_dict_final[method] - 4, method_dict[method]['label'], fontsize=7, fontweight='bold', color='black')
    elif method == 'hBEEFvdW':
        axs[3,1].text(chemisorption_mad_dict_final[method] - 16, physisorption_mad_dict_final[method] - 1, method_dict[method]['label'], fontsize=7, fontweight='bold', color='black')
    elif method == '03_RPBE':
        axs[3,1].text(chemisorption_mad_dict_final[method] - 8.5, physisorption_mad_dict_final[method] - 1, method_dict[method]['label'], fontsize=7)
    else:
        axs[3,1].text(chemisorption_mad_dict_final[method] + 1.5, physisorption_mad_dict_final[method] - 1, method_dict[method]['label'], fontsize=7)

axs[3,1].set_xlim(0, 80)
axs[3,1].set_ylim(0, 80)
axs[3,1].set_xticks(np.arange(0, 81, 20))
axs[3,1].set_yticks(np.arange(0, 81, 20))

axs[3,0].set_xticklabels([])

ax_chem.set_ylim([0,75])
ax_chem.set_yticks([0,25,50,75])
axs[2,0].set_ylim([0,75])
axs[2,0].set_yticks([0,25,50,75])
axs[3,0].set_ylim([0,50])

axs[0,1].axis('off')
axs[1,1].axis('off')
axs[2,1].axis('off')

fig.supylabel('Mean Absolute Deviation (kJ/mol)', fontsize=12)

plt.savefig("Figures/MAIN_Figure-CE39_Comparison.png")

## Parameter optimization

In [107]:
# Find optimal parameters
vasp_method = 'vasp'

ce39_beef_opt_list = ['05','12','15','18','26','33','38'] 
ce39_opt_int_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0, 'BEEFvdW': 0, 'beefx': 0, 'beefc': 0, 'exx': 0, 'exx0':0,'exx1':0,'exx2':0 ,'exx3':0, 'rpac_beef': 0,'nlc': 0, 'rpac': 0, 'rpac_beef':0} for ce39_idx in ce39_beef_opt_list}

for method in ['BEEFvdW','dhBEEFvdW']:
    if method in ['hBEEFvdW','dhBEEFvdW']:
        root_folder = f'Data/CE39/Eint_hBEEFvdW'
        rpa_root_folder = f'Data/CE39/Eint_RPA'
        rpa_beef_root_folder = f'Data/CE39/Eint_RPA_BEEF'
    else:
        root_folder = f'Data/CE39/Eint_{method}'

    for ce39_idx in ce39_opt_int_dict:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)

        total_product_energy = 0
        if method == 'BEEFvdW':
            for product_idx in range(num_products):
                total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR.gz', code='vasp')

            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR.gz', code='vasp')
            ce39_opt_int_dict[ce39_idx]['BEEFvdW'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV


        elif method == 'dhBEEFvdW':
            calc_energy_dict = {
                '00': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
                '01': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
                '02': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
                '03_0': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
                '03_1': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
                '03_2': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
                '03_3': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
            }

            for calc_type in calc_energy_dict:
                for product_idx in range(num_products):
                    calc_energy_dict[calc_type]['total_product_energy'] += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{calc_type}.gz', code='vasp')

                calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
                calc_energy_dict[calc_type]['reactant_surface_energy'] = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
                calc_energy_dict[calc_type]['eint'] = (calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - num_products * calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV

            # RPA regular interaction energy
            total_product_energy = 0
            for product_idx in range(num_products):
                total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz', code='vasp_rpa')

            reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
            reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
            rpac_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV

            # RPA@BEEF interaction energy
            total_product_energy = 0
            for product_idx in range(num_products):
                total_product_energy += get_energy(f'{rpa_beef_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz', code='vasp_rpa')

            reactant_molecule_energy = get_energy(f'{rpa_beef_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
            reactant_surface_energy = get_energy(f'{rpa_beef_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
            rpac_beef_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV


            beefx_eint = calc_energy_dict['02']['eint']
            exx_eint = calc_energy_dict['03_3']['eint']
            nlc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['01']['eint']
            beefc_eint = calc_energy_dict['01']['eint'] - calc_energy_dict['02']['eint']

            ce39_opt_int_dict[ce39_idx]['beefx'] = beefx_eint
            ce39_opt_int_dict[ce39_idx]['beefc'] = beefc_eint
            ce39_opt_int_dict[ce39_idx]['exx'] = exx_eint
            ce39_opt_int_dict[ce39_idx]['exx0'] = calc_energy_dict['03_0']['eint']
            ce39_opt_int_dict[ce39_idx]['exx1'] = calc_energy_dict['03_1']['eint']
            ce39_opt_int_dict[ce39_idx]['exx2'] = calc_energy_dict['03_2']['eint']
            ce39_opt_int_dict[ce39_idx]['exx3'] = calc_energy_dict['03_3']['eint']
            ce39_opt_int_dict[ce39_idx]['nlc'] = nlc_eint
            ce39_opt_int_dict[ce39_idx]['rpac'] = rpac_eint
            ce39_opt_int_dict[ce39_idx]['rpac_beef'] = rpac_beef_eint
            ce39_opt_int_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']


for omega_val in ['0','1','2','3','BEEF']:
    # Make a 2D grid of exx and vdw values
    exx_values = np.linspace(0, 100, 101)
    rpac_values = np.linspace(0, 100, 101)
    X, Y = np.meshgrid(exx_values, rpac_values)
    # Make a set of Z values for each of the ce39 reactions
    Z = np.zeros((101,101,len(ce39_beef_opt_list)))

    idx_counter = 0

    for ce39_idx in ce39_beef_opt_list:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)
        if omega_val == 'BEEF':
            ene_component_dict = {
                'beef_x': ce39_opt_int_dict[ce39_idx]['beefx'],
                'rpa_c': ce39_opt_int_dict[ce39_idx]['rpac_beef'],
                'exx': ce39_opt_int_dict[ce39_idx][f'exx3'],
                'beef_c': ce39_opt_int_dict[ce39_idx]['beefc'],
                'vdw_c': ce39_opt_int_dict[ce39_idx]['nlc'],
            }
        else:
            ene_component_dict = {
                'beef_x': ce39_opt_int_dict[ce39_idx]['beefx'],
                'rpa_c': ce39_opt_int_dict[ce39_idx]['rpac'],
                'exx': ce39_opt_int_dict[ce39_idx][f'exx{omega_val}'],
                'beef_c': ce39_opt_int_dict[ce39_idx]['beefc'],
                'vdw_c': ce39_opt_int_dict[ce39_idx]['nlc'],
            }

        for i in range(X.shape[0]):
            for j in range(X.shape[1]):
                    exx_val = X[i,j]
                    corr_val = Y[i,j]
                    # Get value of adsorption energy for this exx value
                    exx_eads = (1 - exx_val / 100) * ene_component_dict['beef_x'] + exx_val / 100 * ene_component_dict['exx']
                    corr_eads = (1 - corr_val / 100) * (ene_component_dict['beef_c']) + corr_val / 100 * ene_component_dict['rpa_c']   + ene_component_dict['vdw_c']
                    Z[i,j,idx_counter] = exx_eads + corr_eads - ce39_opt_int_dict[ce39_idx]['BEEFvdW'] + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)']

        idx_counter += 1

    # Get system specific MAD grids
    system_MAD_grid = np.zeros((101,101,len(ce39_beef_opt_list)))
    for idx, ce39_idx in enumerate(ce39_beef_opt_list):
        for i in range(X.shape[0]):
            for j in range(X.shape[1]):
                system_MAD_grid[i,j,idx] = abs(Z[i,j,idx] - ce39_opt_int_dict[ce39_idx]['Experiment']) * ce39_dataset_dict[ce39_idx]['MAD Weight']

    # Calculate the MAD for each point in the grid
    EXP_MAD_grid = np.zeros((101,101))
    # Get the minimum error for each reaction to normalize the differences
    ce39_min_err_dict = {}
    ce39_err_range_dict = {}
    for idx, ce39_idx in enumerate(ce39_beef_opt_list):
        ce39_idx_err_array = abs(Z[:,:,idx] - ce39_opt_int_dict[ce39_idx]['Experiment'])
        ce39_idx_err_min = np.min(ce39_idx_err_array)
        ce39_idx_err_max = np.max(ce39_idx_err_array)
        ce39_min_err_dict[ce39_idx] = ce39_idx_err_min
        ce39_err_range_dict[ce39_idx] = ce39_idx_err_max - ce39_idx_err_min
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            mad_sum = 0
            weight_sum = 0
            for idx, ce39_idx in enumerate(ce39_beef_opt_list):
                # Get the range of differences for this reaction
                diff = abs(Z[i,j,idx] - ce39_opt_int_dict[ce39_idx]['Experiment']) 

                mad_sum += diff * ce39_dataset_dict[ce39_idx]['MAD Weight']
                weight_sum += 1
            EXP_MAD_grid[i,j] = mad_sum/weight_sum

    # Print minimum MAD values for EXP_MAD_grid
    exp_min_mad = np.min(EXP_MAD_grid)
    exp_min_idx = np.unravel_index(np.argmin(EXP_MAD_grid), EXP_MAD_grid.shape)
    exp_optimal_exx = X[exp_min_idx]
    exp_optimal_vdw = Y[exp_min_idx]
    exp_min_mad_hybrid = np.min(EXP_MAD_grid[0,:])
    exp_min_idx_hybrid = np.unravel_index(np.argmin(EXP_MAD_grid[0,:]), EXP_MAD_grid[0,:].shape)
    exp_optimal_exx_hybrid = X[0,exp_min_idx_hybrid]

    contour_levels = np.linspace(10, 38, 101)

    # Plot the MAD grid for both RPA and EXP
    fig, axs = plt.subplots(2,1,figsize=(3.365, 5), dpi=450, sharex=True, height_ratios=[0.3,0.6],tight_layout=True)

    axs[0].plot(X[0,:],EXP_MAD_grid[0,:], 'b-')

    # axs[0].set_ylim([10,100])

    axs[0].plot(exp_optimal_exx_hybrid, exp_min_mad_hybrid , 'go', markersize=7)
    axs[0].text(exp_optimal_exx_hybrid+3, exp_min_mad_hybrid - 5, f'EXX={float(exp_optimal_exx_hybrid):.0f}%, RPA=0%\nMAD={exp_min_mad_hybrid:.1f} kJ/mol', color='white', fontsize=8, bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.3'))


    # axs[0].set_yscale('log')

    axs[0].set_ylabel('CE7 MAD (kJ/mol)')

    axs[0].set_yticks([0,10,20,30,40,70,100])
    axs[0].set_yticklabels(['0','10','20','30','40','70','100'])

    axs[0].set_ylim([0,100])

    # Remove log scale minor ticks
    axs[0].yaxis.set_minor_locator(plt.NullLocator())

    axs[1].contourf(X, Y, EXP_MAD_grid, levels=contour_levels, cmap='cividis')
    axs[1].plot(exp_optimal_exx, exp_optimal_vdw, 'ro', markersize=7, label=f'EXX={exp_optimal_exx:.0f}%, RPA={exp_optimal_vdw:.0f}%, MAD={exp_min_mad:.1f} kJ/mol')
    axs[1].text(exp_optimal_exx+1, exp_optimal_vdw + 10, f'EXX={exp_optimal_exx:.0f}%, RPA={exp_optimal_vdw:.0f}%\nMAD={exp_min_mad:.1f} kJ/mol', color='white', fontsize=8, bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.3'))

    axs[1].plot(0, 0, 'bo', markersize=7)
    axs[1].text(5, 0 +  10, f'EXX=0%, RPA=0%\nMAD={EXP_MAD_grid[0,0]:.1f} kJ/mol', color='white', fontsize=8, bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.3'))

    axs[1].set_xlim([0,100])
    axs[1].set_ylim([60,0])

    # Move xticks to top
    axs[1].xaxis.set_ticks_position('top')
    axs[1].xaxis.set_label_position('top')

    # axs[1].set_title('Experiment')
    axs[1].set_xlabel('EXX (%)')
    axs[1].set_ylabel('RPA correlation (%)')

    # Add colorbar to the top of the first plot
    cbar = fig.colorbar(
        axs[1].collections[0],
        ax=axs[1],
        orientation='horizontal',
        pad=0.12
    )
    cbar.set_label('CE7 MAD (kJ/mol)') 
    # Set colorbar ticks
    cbar.set_ticks(np.arange(10,39, 4))

    if omega_val == '3':
        axs[0].set_title(r'\textbf{FINAL} --- $\omega = 0.3$ \AA{}$^{-1}$', fontsize=10)
        plt.savefig('Figures/SI_Figure-BEEF_XC_optimize_w3_FINAL.png')
    elif omega_val == 'BEEF':
        axs[0].set_title(r'RPA@BEEF-vdW with $\omega = 0.3$ \AA{}$^{-1}$', fontsize=10)
        plt.savefig('Figures/SI_Figure-BEEF_XC_optimize_BEEF.png')
    else:
        axs[0].set_title(f'$\\omega = 0.{omega_val}$' + r' \AA{}$^{-1}$', fontsize=10)
        plt.savefig(f'Figures/SI_Figure-BEEF_XC_optimize_w{omega_val}.png')


/var/folders/zp/fz7h39h554nb51_blfh6qmqw0000gp/T/ipykernel_31984/4155955267.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  axs[0].text(exp_optimal_exx_hybrid+3, exp_min_mad_hybrid - 5, f'EXX={float(exp_optimal_exx_hybrid):.0f}%, RPA=0%\nMAD={exp_min_mad_hybrid:.1f} kJ/mol', color='white', fontsize=8, bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.3'))
/var/folders/zp/fz7h39h554nb51_blfh6qmqw0000gp/T/ipykernel_31984/4155955267.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  axs[0].text(exp_optimal_exx_hybrid+3, exp_min_mad_hybrid - 5, f'EXX={float(exp_optimal_exx_hybrid):.0f}%, RPA=0%\nMAD={exp_min_mad_hybrid:.1

# Adsorpion configuration

## CO on Pt(111), Cu(111), Rh(111) and Pd(111)

In [108]:
# Get layer effect
layer_rel_ene_dict = {x: {method: 0  for method in ['4L','5L','Rel']} for x in ['Cu','Pt','Rh','Pd']}

root_folder = 'Data/CO_Ads/Geom_Effect'

for metal in ['Cu','Pt','Rh','Pd']:
    atop_energy = get_energy(f'{root_folder}/{metal}111_4L/ATOP/OUTCAR_00.gz',code='vasp')
    fcc_energy = get_energy(f'{root_folder}/{metal}111_4L/FCC/OUTCAR_00.gz',code='vasp')
    layer_rel_ene_dict[metal]['4L'] = (atop_energy - fcc_energy) * 1000 / kjmol_to_meV
    atop_energy = get_energy(f'{root_folder}/{metal}111_5L/ATOP/OUTCAR_00.gz',code='vasp')
    fcc_energy = get_energy(f'{root_folder}/{metal}111_5L/FCC/OUTCAR_00.gz',code='vasp')
    layer_rel_ene_dict[metal]['5L'] = (atop_energy - fcc_energy) * 1000 / kjmol_to_meV
    layer_rel_ene_dict[metal]['Rel'] = layer_rel_ene_dict[metal]['5L'] - layer_rel_ene_dict[metal]['4L']


co_ads_dict = {x: {method: 0  for method in ['hBEEFvdW','dhBEEFvdW','RPA','BEEFvdW'] + dft_xc_list} for x in ['Cu','Pt','Rh','Pd']}

for method in ['hBEEFvdW','dhBEEFvdW', 'RPA', 'BEEFvdW']:
    for metal in ['Cu','Pt','Rh','Pd']:
        if method == 'RPA':
            root_folder = f'Data/CO_Ads/{method}/{metal}111'
            fcc_energy = get_energy(f'{root_folder}_FCC/OUTCAR_02_EXX.gz', code='vasp') + get_energy(f'{root_folder}_FCC/OUTCAR_04_RPA.gz', code='vasp_rpa')
            atop_energy = get_energy(f'{root_folder}_ATOP/OUTCAR_02_EXX.gz', code='vasp') + get_energy(f'{root_folder}_ATOP/OUTCAR_04_RPA.gz', code='vasp_rpa')
            relative_energy = (atop_energy - fcc_energy)*1000 / kjmol_to_meV
            co_ads_dict[metal]['RPA'] = relative_energy + layer_rel_ene_dict[metal]['Rel']
        elif method == 'BEEFvdW':
            root_folder = f'Data/CO_Ads/hBEEFvdW/{metal}111'
            fcc_energy = get_energy(f'{root_folder}_FCC/OUTCAR_00.gz', code='vasp')
            atop_energy = get_energy(f'{root_folder}_ATOP/OUTCAR_00.gz', code='vasp')
            relative_energy = (atop_energy - fcc_energy)*1000 / kjmol_to_meV
            co_ads_dict[metal]['BEEFvdW'] = relative_energy + layer_rel_ene_dict[metal]['Rel']
        elif method == 'hBEEFvdW':
            root_folder = f'Data/CO_Ads/hBEEFvdW/{metal}111'
            calc_energy_dict = {
                '00': {'fcc_energy': 0, 'atop_energy': 0},
                '01': {'fcc_energy': 0, 'atop_energy': 0},
                '02': {'fcc_energy': 0, 'atop_energy': 0},
                '03_3': {'fcc_energy': 0, 'atop_energy': 0}
            }        

            for calc_type in calc_energy_dict:
                calc_energy_dict[calc_type]['fcc_energy'] = get_energy(f'{root_folder}_FCC/OUTCAR_{calc_type}.gz', code='vasp')
                calc_energy_dict[calc_type]['atop_energy'] = get_energy(f'{root_folder}_ATOP/OUTCAR_{calc_type}.gz', code='vasp')

            beefx_fcc = calc_energy_dict['02']['fcc_energy']
            beefx_atop = calc_energy_dict['02']['atop_energy']
            exx_fcc = calc_energy_dict['03_3']['fcc_energy']
            exx_atop = calc_energy_dict['03_3']['atop_energy']
            beefc_fcc = calc_energy_dict['00']['fcc_energy'] - calc_energy_dict['02']['fcc_energy']
            beefc_atop = calc_energy_dict['00']['atop_energy'] - calc_energy_dict['02']['atop_energy']

            frac = 0.175
            fcc_energy = frac * exx_fcc + (1-frac)*beefx_fcc + beefc_fcc
            atop_energy = frac * exx_atop + (1-frac)*beefx_atop + beefc_atop

            relative_energy = (atop_energy - fcc_energy)*1000 / kjmol_to_meV
            co_ads_dict[metal]['hBEEFvdW'] = relative_energy + layer_rel_ene_dict[metal]['Rel']
        elif method == 'dhBEEFvdW':
            root_folder = f'Data/CO_Ads/hBEEFvdW/{metal}111'
            rpa_root_folder = f'Data/CO_Ads/RPA/{metal}111'
            calc_energy_dict = {
                '00': {'fcc_energy': 0, 'atop_energy': 0},
                '01': {'fcc_energy': 0, 'atop_energy': 0},
                '02': {'fcc_energy': 0, 'atop_energy': 0},
                '03_3': {'fcc_energy': 0, 'atop_energy': 0}
            }        

            for calc_type in calc_energy_dict:
                calc_energy_dict[calc_type]['fcc_energy'] = get_energy(f'{root_folder}_FCC/OUTCAR_{calc_type}.gz', code='vasp')
                calc_energy_dict[calc_type]['atop_energy'] = get_energy(f'{root_folder}_ATOP/OUTCAR_{calc_type}.gz', code='vasp')

            rpac_fcc = get_energy(f'{rpa_root_folder}_FCC/OUTCAR_04_RPA.gz', code='vasp_rpa')
            rpac_atop = get_energy(f'{rpa_root_folder}_ATOP/OUTCAR_04_RPA.gz', code='vasp_rpa')

            beefx_fcc = calc_energy_dict['02']['fcc_energy']
            beefx_atop = calc_energy_dict['02']['atop_energy']
            exx_fcc = calc_energy_dict['03_3']['fcc_energy']
            exx_atop = calc_energy_dict['03_3']['atop_energy']
            nlc_fcc = calc_energy_dict['00']['fcc_energy'] - calc_energy_dict['01']['fcc_energy']
            nlc_atop = calc_energy_dict['00']['atop_energy'] - calc_energy_dict['01']['atop_energy']
            beefc_fcc = calc_energy_dict['01']['fcc_energy'] - calc_energy_dict['02']['fcc_energy']
            beefc_atop = calc_energy_dict['01']['atop_energy'] - calc_energy_dict['02']['atop_energy']

            x_frac = 0.25
            c_frac = 0.15
            fcc_energy = x_frac * exx_fcc + (1-x_frac)*beefx_fcc + c_frac*rpac_fcc + (1-c_frac)*beefc_fcc + nlc_fcc
            atop_energy = x_frac * exx_atop + (1-x_frac)*beefx_atop + c_frac*rpac_atop + (1-c_frac)*beefc_atop + nlc_atop
            relative_energy = (atop_energy - fcc_energy)*1000 / kjmol_to_meV
            co_ads_dict[metal]['dhBEEFvdW'] = relative_energy + layer_rel_ene_dict[metal]['Rel']

for dft_xc in dft_xc_list:
    for metal in ['Cu','Pt','Rh','Pd']:
        root_folder = f'Data/CO_Ads/DFT/{metal}111'
        fcc_energy = get_energy(f'{root_folder}_FCC/OUTCAR_{dft_xc}.gz', code='vasp')
        atop_energy = get_energy(f'{root_folder}_ATOP/OUTCAR_{dft_xc}.gz', code='vasp')
        relative_energy = (atop_energy - fcc_energy)*1000 / kjmol_to_meV
        co_ads_dict[metal][dft_xc] = relative_energy + layer_rel_ene_dict[metal]['Rel']

In [109]:
# Make a DataFrame from the dictionary
co_ads_df = pd.DataFrame.from_dict(co_ads_dict, orient='index')
# Add an additional row at the bottom that is the MAD for each method
co_ads_df = co_ads_df.round(1)
co_ads_df.columns = [method_dict[method]['label'] for method in co_ads_df.columns.tolist()]
display(co_ads_df.T)

convert_df_to_latex_input(
    df=co_ads_df.T,
    start_input = '\\begin{table}[h]\n',
    label = "tab:co_ads",
    caption = r"Relative adsorption energies of CO at the atop and fcc sites on Cu(111), Pt(111), Rh(111), and Pd(111) calculated with hBEEF-vdW@BEEF-vdW, dhBEEF-vdW@BEEF-vdW, RPA@PBE, BEEF-vdW and selection of density functional approximations. All values are in kJ/mol.",
    replace_input= {
        r'Cu': r'Cu(111)',
        r'Pt': r'Pt(111)',
        r'Rh': r'Rh(111)',
        r'Pd': r'Pd(111)'
    },
    float_fmt="%.1f",
    filename = "Tables/SI_Table-CO_Adsorption_Energies.tex"
)

,Cu,Pt,Rh,Pd
hBEEF-vdW,-8.1,-0.6,-18.9,50.4
dhBEEF-vdW,-1.7,-5.3,-11.9,49.1
RPA@PBE,-14.5,-47.8,-27.6,87.7
BEEF-vdW,2.7,4.9,-11.1,44.7
PBE,15.6,8.1,-6.9,53.2
PBEsol,25.8,14.6,7.5,64.2
RPBE,8.3,2.9,-15.9,45.7
revPBE,9.7,3.5,-14.2,46.9
BLYP,1.1,-1.9,-28.4,36.9
r$^2$SCAN,13.9,3.6,-9.2,56.2


## Effect of EXX and RPA correlation

In [110]:
# Make a 2D grid of exx and vdw values
exx_values = np.linspace(0, 100, 101)
rpac_values = np.linspace(0.0, 100, 101)
X, Y = np.meshgrid(exx_values, rpac_values)
Z = np.zeros((101,101))

idx_counter = 0

root_folder = f'Data/CO_Ads/hBEEFvdW/Pt111'
rpa_root_folder = f'Data/CO_Ads/RPA/Pt111'
calc_energy_dict = {
    '00': {'fcc_energy': 0, 'atop_energy': 0},
    '01': {'fcc_energy': 0, 'atop_energy': 0},
    '02': {'fcc_energy': 0, 'atop_energy': 0},
    '03_3': {'fcc_energy': 0, 'atop_energy': 0}
}        

for calc_type in calc_energy_dict:
    calc_energy_dict[calc_type]['fcc_energy'] = get_energy(f'{root_folder}_FCC/OUTCAR_{calc_type}.gz', code='vasp')
    calc_energy_dict[calc_type]['atop_energy'] = get_energy(f'{root_folder}_ATOP/OUTCAR_{calc_type}.gz', code='vasp')

rpac_fcc = get_energy(f'{rpa_root_folder}_FCC/OUTCAR_04_RPA.gz', code='vasp_rpa')
rpac_atop = get_energy(f'{rpa_root_folder}_ATOP/OUTCAR_04_RPA.gz', code='vasp_rpa')

beefx_fcc = calc_energy_dict['02']['fcc_energy']
beefx_atop = calc_energy_dict['02']['atop_energy']
exx_fcc = calc_energy_dict['03_3']['fcc_energy']
exx_atop = calc_energy_dict['03_3']['atop_energy']
nlc_fcc = calc_energy_dict['00']['fcc_energy'] - calc_energy_dict['01']['fcc_energy']
nlc_atop = calc_energy_dict['00']['atop_energy'] - calc_energy_dict['01']['atop_energy']
beefc_fcc = calc_energy_dict['01']['fcc_energy'] - calc_energy_dict['02']['fcc_energy']
beefc_atop = calc_energy_dict['01']['atop_energy'] - calc_energy_dict['02']['atop_energy']

relative_energy = (atop_energy - fcc_energy)*1000 / kjmol_to_meV
co_ads_dict[metal]['hBEEFvdW'] = relative_energy

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        exx_val = X[i,j]
        corr_val = Y[i,j]
        fcc_energy = exx_val/100 * exx_fcc + (1-exx_val/100)*beefx_fcc + corr_val/100*rpac_fcc + (1-corr_val/100)*beefc_fcc + nlc_fcc
        atop_energy = exx_val/100 * exx_atop + (1-exx_val/100)*beefx_atop + corr_val/100*rpac_atop + (1-corr_val/100)*beefc_atop + nlc_atop
        Z[i,j] = (atop_energy - fcc_energy)*1000 / kjmol_to_meV + layer_rel_ene_dict[metal]['Rel']


# Only include negative Z values
Z = np.where(Z < 0, Z, np.nan)
fig, axs = plt.subplots( figsize=(4,4), dpi=450, constrained_layout=True)

contour1 = axs.contourf(X, Y, Z, levels=100, cmap='viridis', vmin=-50, vmax=0)

fig.colorbar(contour1, ax=axs, label='Relative (Top - Hollow) energy (kJ/mol)')
axs.set_title('Top - Hollow Site Energy Difference', fontsize=10)
axs.set_xlabel('EXX scaling factor (\%)')
axs.set_ylabel('RPA@PBE correlation scaling factor (\%)')

plt.savefig(f"Figures/SI_Figure-CO_Adsorption_EXX_Pt111.png")

## CO adsorption GW DOS

In [111]:
# Plot the effect on the DOS
final_dos_dict = {
    'GW': np.loadtxt('Data/CO_Ads/DOS/DOS_GW.txt',skiprows=1),
    'BEEF-vdW': np.loadtxt('Data/CO_Ads/DOS/DOS_BEEF.txt',skiprows=1),
    'BEEF10-vdW': np.loadtxt('Data/CO_Ads/DOS/DOS_BEEF10.txt',skiprows=1),
    'BEEF25-vdW': np.loadtxt('Data/CO_Ads/DOS/DOS_BEEF25.txt',skiprows=1)
}

fig, axs = plt.subplots(4, 1, figsize=(3.365,5), dpi=450, constrained_layout=True, sharex=True,sharey=True)
for i, (method, dos_data) in enumerate(final_dos_dict.items()):
    axs[i].plot(dos_data[:,0], dos_data[:,1], color='grey', label='Pt(d)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,1], color='grey', alpha=0.1)
    axs[i].plot(dos_data[:,0], dos_data[:,2], color='green', label='C(p)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,2], color='green', alpha=0.1)
    axs[i].axvline(0, color='black', linestyle='--', linewidth=0.8)
    if method == 'BEEF10-vdW':
        axs[i].set_title(f'h(10%)BEEF-vdW@BEEF-vdW')
    elif method == 'BEEF25-vdW':
        axs[i].set_title(f'h(25%)BEEF-vdW@BEEF-vdW')
    else:
        axs[i].set_title(f'{method}')
    axs[i].set_ylabel('DOS (states/eV)') 
    axs[i].axvline(4.3, color='red', linestyle='--', linewidth=0.8, label=r'2$\pi^*$')

axs[1].legend()

axs[3].set_xlabel('$E-E_f$ (eV)')
axs[0].set_xlim([-10,10])
axs[0].set_ylim([0,2])

plt.savefig('Figures/SI_Figure-CO_Adsorption_DOS_Pt111.png')

In [112]:
# Plot convergence of the GW DOS with respect to k-points

gw_convergence_kpoint_dict = {
    5: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_kpoints_5.txt',skiprows=1),
    9: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_kpoints_9.txt',skiprows=1),
    11: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_kpoints_11.txt',skiprows=1)
}

fig, axs = plt.subplots(3, 1, figsize=(3.365,5), dpi=450, constrained_layout=True,sharex=True,sharey=True)

for i, (kpoints, dos_data) in enumerate(gw_convergence_kpoint_dict.items()):
    axs[i].plot(dos_data[:,0], dos_data[:,1], color='blue', label='Pt(d)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,1], color='blue', alpha=0.1)
    axs[i].plot(dos_data[:,0], dos_data[:,2], color='red', label='C(p)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,2], color='red', alpha=0.1)
    axs[i].axvline(0, color='black', linestyle='--', linewidth=0.8)
    axs[i].set_title(f'GW k-points: {kpoints}x{kpoints}x1')
    if i == 0:
        axs[i].legend()
fig.supylabel('DOS (states/eV)')
axs[2].set_xlabel('$E-E_f$ (eV)')
axs[0].set_xlim([-10,10])
axs[0].set_ylim([0,2])

plt.savefig('Figures/SI_Figure-DOS_GW_kpoint_convergence.png')

In [113]:
# Plot GW ENCUT convergence

gw_convergence_encut_dict = {
    150: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_encut_150.txt',skiprows=1),
    310: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_encut_310.txt',skiprows=1),
    400: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_encut_400.txt',skiprows=1)
}

fig, axs = plt.subplots(3, 1, figsize=(3.365,5), dpi=450, constrained_layout=True,sharex=True,sharey=True)

for i, (encut, dos_data) in enumerate(gw_convergence_encut_dict.items()):
    axs[i].plot(dos_data[:,0], dos_data[:,1], color='blue', label='Pt(d)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,1], color='blue', alpha=0.1)
    axs[i].plot(dos_data[:,0], dos_data[:,2], color='red', label='C(p)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,2], color='red', alpha=0.1)
    axs[i].axvline(0, color='black', linestyle='--', linewidth=0.8)
    axs[i].set_title(f'GW ENCUT: {encut} eV')
    if i == 0:
        axs[i].legend()
fig.supylabel('DOS (states/eV)')
axs[2].set_xlabel('$E-E_f$ (eV)')
axs[0].set_xlim([-10,10])
axs[0].set_ylim([0,2])

plt.savefig('Figures/SI_Figure-DOS_GW_encut_convergence.png')

In [114]:
# Plot convergence of the GW DOS with respect to number of bands

gw_convergence_nbands_dict = {
    128: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_nbands_128.txt',skiprows=1),
    256: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_nbands_256.txt',skiprows=1),
    384: np.loadtxt('Data/CO_Ads/DOS/DOS_GW_Convergence_nbands_384.txt',skiprows=1)
}

fig, axs = plt.subplots(3, 1, figsize=(3.365,5), dpi=450, constrained_layout=True,sharex=True,sharey=True)

for i, (nbands, dos_data) in enumerate(gw_convergence_nbands_dict.items()):
    axs[i].plot(dos_data[:,0], dos_data[:,1], color='blue', label='Pt(d)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,1], color='blue', alpha=0.1)
    axs[i].plot(dos_data[:,0], dos_data[:,2], color='red', label='C(p)',linewidth=1)
    axs[i].fill_between(dos_data[:,0], dos_data[:,2], color='red', alpha=0.1)
    axs[i].axvline(0, color='black', linestyle='--', linewidth=0.8)
    axs[i].set_title(f'GW NBANDS: {nbands}')
    if i == 0:
        axs[i].legend()
fig.supylabel('DOS (states/eV)')
axs[2].set_xlabel('$E-E_f$ (eV)')
axs[0].set_xlim([-10,10])
axs[0].set_ylim([0,2])

plt.savefig('Figures/SI_Figure-DOS_GW_nbands_convergence.png')

## Graphene on Ni(111)

In [115]:
# Get the geometry relaxation effect to convert Eint to Eads with BEEF-vdW
relaxed_beef_eads = (get_energy('Data/Gr_Ads/Geom_Effect/GrNi111/OUTCAR.gz', code='vasp') - get_energy('Data/Gr_Ads/Geom_Effect/Ni111/OUTCAR.gz', code='vasp') - get_energy('Data/Gr_Ads/Geom_Effect/Gr/OUTCAR.gz', code='vasp'))*1000/8

dft_zlist = [int(x) for x in "170 180 190 200 204 208 212 216 220 224 230 240 250 260 270 280 290 310 320 400 450 500".split()]

# Get the relaxation effect from BEEF-vdW
root_folder = f'Data/Gr_Ads/hBEEFvdW'
zheight_angstrom = np.array(dft_zlist) / 100
beef_energies = np.array([get_energy(f'{root_folder}/{zheight}/OUTCAR_00.gz', code='vasp') - get_energy(f'{root_folder}/500/OUTCAR_00.gz', code='vasp') for zheight in dft_zlist])*1000/8
beef_spline = UnivariateSpline(zheight_angstrom, beef_energies, s=4) 

# Calculate the geometry relaxation effect
delta_geom_eads = relaxed_beef_eads - np.min(beef_spline([zheight/1000 for zheight in range(2000,2500)]))

grni111_dft_eads_dict = {z: {'BEEF-vdW': 0} for z in dft_zlist}

for method in ['BEEF-vdW']:
    for zheight in dft_zlist:
        root_folder = f'Data/Gr_Ads/hBEEFvdW'
        if method == 'BEEF-vdW':
            grni111_dft_eads_dict[zheight][method] = (get_energy(f'{root_folder}/{zheight}/OUTCAR_00.gz', code='vasp') - get_energy(f'{root_folder}/500/OUTCAR_00.gz', code='vasp'))*1000/8 + delta_geom_eads

rpa_zlist = [190,200,208,216,224,250,260,280,310,320,400,500]
grni111_rpa_eads_dict = {z: { 'RPA': 0, 'dhBEEF-vdW': 0} for z in rpa_zlist}

for method in ['RPA', 'dhBEEF-vdW']:
    for zheight in rpa_zlist:
        rpa_root_folder = f'Data/Gr_Ads/RPA'
        if method == 'RPA':
            grni111_rpa_eads_dict[zheight][method] = (get_energy(f'{rpa_root_folder}/{zheight}/OUTCAR_02_EXX.gz', code='vasp') + get_energy(f'{rpa_root_folder}/{zheight}/OUTCAR_04_RPA.gz', code='vasp_rpa') - get_energy(f'{rpa_root_folder}/500/OUTCAR_02_EXX.gz', code='vasp') - get_energy(f'{rpa_root_folder}/500/OUTCAR_04_RPA.gz', code='vasp_rpa'))*1000/8 + delta_geom_eads
        elif method == 'dhBEEF-vdW':
            root_folder = f'Data/Gr_Ads/hBEEFvdW'
            calc_energy_dict = {
                '00': {'eint': 0},
                '01': {'eint': 0},
                '02': {'eint': 0},
                '03_3': {'eint': 0}
            }
            for calc_type in calc_energy_dict:
                calc_energy_dict[calc_type]['eint'] = (get_energy(f'{root_folder}/{zheight}/OUTCAR_{calc_type}.gz', code='vasp') - get_energy(f'{root_folder}/500/OUTCAR_{calc_type}.gz', code='vasp'))*1000/8
            rpac_eint = (get_energy(f'{rpa_root_folder}/{zheight}/OUTCAR_04_RPA.gz', code='vasp_rpa') -  get_energy(f'{rpa_root_folder}/500/OUTCAR_04_RPA.gz', code='vasp_rpa'))*1000/8
            beefx_eint = calc_energy_dict['02']['eint']
            exx_eint = calc_energy_dict['03_3']['eint']
            nlc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['01']['eint']
            beefc_eint = calc_energy_dict['01']['eint'] - calc_energy_dict['02']['eint']

            x_frac = 0.25
            c_frac = 0.15
            grni111_rpa_eads_dict[zheight][method] = x_frac * exx_eint + (1-x_frac)*beefx_eint + c_frac*rpac_eint + (1-c_frac)*beefc_eint + nlc_eint + delta_geom_eads

hbeef_zlist = [int(x) for x in "170 180 190 200 204 208 212 216 220 230 240 250 260 270 280 290 310 320 400 450 500".split()]
grni111_hbeef_eads_dict = {z: { 'hBEEF-vdW': 0} for z in hbeef_zlist}

for method in ['hBEEF-vdW']:
    for zheight in hbeef_zlist:
        root_folder = f'Data/Gr_Ads/hBEEFvdW'
        calc_energy_dict = {
            '00': {'eint': 0},
            '01': {'eint': 0},
            '02': {'eint': 0},
            '03_3': {'eint': 0}
        }        

        for calc_type in calc_energy_dict:
            calc_energy_dict[calc_type]['eint'] = (get_energy(f'{root_folder}/{zheight}/OUTCAR_{calc_type}.gz', code='vasp')  - get_energy(f'{root_folder}/500/OUTCAR_{calc_type}.gz', code='vasp'))*1000/8

        beefx_eint = calc_energy_dict['02']['eint']
        exx_eint = calc_energy_dict['03_3']['eint']
        beefc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['02']['eint']

        frac = 0.175
        grni111_hbeef_eads_dict[zheight][method] = frac * exx_eint + (1-frac)*beefx_eint + beefc_eint  + delta_geom_eads


dft_xc_zlist = [int(x) for x in "170 180 190 208 216 220 224 230 240 250 260 270 280 290 310 320 340 370 400 500".split()]
grni111_dft_xc_eads_dict = {z: {dft_xc: 0  for dft_xc in dft_xc_list} for z in dft_xc_zlist}

for dft_xc in dft_xc_list:
    for zheight in dft_xc_zlist:
        root_folder = f'Data/Gr_Ads/DFT'
        grni111_dft_xc_eads_dict[zheight][dft_xc] = (get_energy(f'{root_folder}/{zheight}/OUTCAR_{dft_xc}.gz', code='vasp') - get_energy(f'{root_folder}/500/OUTCAR_{dft_xc}.gz', code='vasp'))*1000/8 + delta_geom_eads



In [116]:
# Plot graphene Ni111 adsorption energy curves for different DFT functionals

fig, axs = plt.subplots(1,1,figsize=(6,5), dpi=450, constrained_layout=True)

for dft_xc in dft_xc_list:
    if dft_xc == '07_MS2':
        continue
    dft_zheight_angstrom = np.array(dft_xc_zlist) / 100
    dft_energies = np.array([grni111_dft_xc_eads_dict[z][dft_xc] for z in dft_xc_zlist])
    axs.scatter(dft_zheight_angstrom, dft_energies  / kjmol_to_meV, label=method_dict[dft_xc]['label'], color=color_dict[method_dict[dft_xc]['color']],marker='x')
    dft_xfit = np.linspace(min(dft_zheight_angstrom), max(dft_zheight_angstrom), 100)
    dft_yfit = UnivariateSpline(dft_zheight_angstrom, dft_energies, s=4)(dft_xfit)  / kjmol_to_meV 
    axs.plot(dft_xfit, dft_yfit, linestyle='--',color=color_dict[method_dict[dft_xc]['color']])


# Experimental bounds
x_exp_min, x_exp_max = 2.04, 2.18
y_exp_min, y_exp_max = -11.2, -7.2

# Square outline
axs.plot(
    [x_exp_min, x_exp_max, x_exp_max, x_exp_min, x_exp_min],
    [y_exp_min, y_exp_min, y_exp_max, y_exp_max, y_exp_min],
    color='black',
    linewidth=1.0,
    label='Experiment'
)

# Light fill inside the square
axs.fill(
    [x_exp_min, x_exp_max, x_exp_max, x_exp_min],
    [y_exp_min, y_exp_min, y_exp_max, y_exp_max],
    color='gray',
    alpha=0.3,
    edgecolor=None
)

# Experimental center
x0 = (x_exp_min + x_exp_max) / 2
y0 = (y_exp_min + y_exp_max) / 2

# Experimental half-widths (errors)
xerr = (x_exp_max - x_exp_min) / 2
yerr = (y_exp_max - y_exp_min) / 2

axs.errorbar(
    x0, y0,
    xerr=xerr,
    yerr=yerr,
    fmt='s',              # square marker
    color='black',
    ecolor='black',
    elinewidth=1.0,
    capsize=3,
    markersize=5,
    zorder=6,
    label='_nolegend_'    # legend already handled by box
)

axs.set_ylim([-25,15])
axs.set_xlim([1.6,5.0])

axs.legend(ncol=2)

axs.set_ylabel('Adsorption energy per C atom (kJ/mol)')
axs.set_xlabel('Distance between graphene and Ni(111) (Å)')
plt.savefig("Figures/SI_Figure-Gr_Ni111_Adsorption_Energy_Curves.png")

In [117]:
fig, axs = plt.subplots(1,2, figsize=(6.695, 3.7),dpi=600,constrained_layout=True)

num_sys = 3
num_xc = len(co_ads_df.columns)

# Make bar positions
bar_width = 0.4
bar_positions = np.arange(num_sys)*(1 + bar_width*num_xc)
spacing = 3.2

system_ordering = ['Cu','Rh','Pt'] 

# Plot the CO adsorption energies

axs[0].barh(np.arange(num_sys)*spacing+2.0, [co_ads_df.loc[element,method_dict['01_PBE']['label']] for element in system_ordering], label='PBE', height=0.4, alpha=1.0,color=color_dict[method_dict['01_PBE']['color']])
axs[0].barh(np.arange(num_sys)*spacing+1.6, [co_ads_df.loc[element,method_dict['14_r2SCANrVV10']['label']] for element in system_ordering], label=r'r$^2$SCAN+rVV10', height=0.4, alpha=1.0,color=color_dict[method_dict['14_r2SCANrVV10']['color']])
axs[0].barh(np.arange(num_sys)*spacing+1.2, [co_ads_df.loc[element,method_dict['BEEFvdW']['label']] for element in system_ordering], color=color_dict[method_dict['BEEFvdW']['color']], label='BEEF-vdW', height=0.4, alpha=1.0)
axs[0].barh(np.arange(num_sys)*spacing+0.8, [co_ads_df.loc[element,method_dict['hBEEFvdW']['label']] for element in system_ordering], color=color_dict[method_dict['hBEEFvdW']['color']], label='hBEEF-vdW', height=0.4, alpha=1.0)
axs[0].barh(np.arange(num_sys)*spacing+0.4, [co_ads_df.loc[element,method_dict['dhBEEFvdW']['label']] for element in system_ordering], color=color_dict[method_dict['dhBEEFvdW']['color']], label='dhBEEF-vdW', height=0.4, alpha=1.0)
axs[0].barh(
    np.arange(num_sys) * spacing,
    [
        co_ads_df.loc[element, method_dict['RPA']['label']]
        if co_ads_df.loc[element, method_dict['RPA']['label']] > -20
        else -20
        for element in system_ordering
    ],
    color=color_dict[method_dict['RPA']['color']],
    label='RPA@PBE',
    height=0.4
)

# Write text labels
for i, element in enumerate(system_ordering):
    if element == 'Rh':
        axs[0].text(-20, i*spacing + 1.2, f"{element}(111)", fontsize=10, fontweight='bold', color='black')
    else:
        axs[0].text(-20, i*spacing + 0.6, f"{element}(111)", fontsize=10, fontweight='bold', color='black')


for i, element in enumerate(system_ordering):
    axs[0].text(-1, i*spacing, f"{co_ads_df.loc[element,method_dict['RPA']['label']]:.1f}", va='center', ha='right', fontsize=7, color='black', fontweight='bold')
    axs[0].text(-1, i*spacing + 0.4, f"{co_ads_df.loc[element,method_dict['dhBEEFvdW']['label']]:.1f}", va='center', ha='right', fontsize=7, color='black', fontweight='bold')
    axs[0].text(-1, i*spacing + 0.8, f"{co_ads_df.loc[element,method_dict['hBEEFvdW']['label']]:.1f}", va='center', ha='right', fontsize=7, color='black', fontweight='bold')
    axs[0].text(-1, i*spacing + 1.2, f"{co_ads_df.loc[element,method_dict['BEEFvdW']['label']]:.1f}", va='center', ha='right', fontsize=7, color='black', fontweight='bold')
    axs[0].text(-1, i*spacing + 1.6, f"{co_ads_df.loc[element,method_dict['01_PBE']['label']]:.1f}", va='center', ha='right', fontsize=7, color='black', fontweight='bold')
    axs[0].text(-1, i*spacing + 2.0, f"{co_ads_df.loc[element,method_dict['14_r2SCANrVV10']['label']]:.1f}", va='center', ha='right', fontsize=7, color='black', fontweight='bold')



zheight_angstrom = np.array(list(grni111_dft_eads_dict.keys())) / 100
beef_energies = np.array([grni111_dft_eads_dict[z]['BEEF-vdW'] for z in dft_zlist])
hbeef_zheight_angstrom = np.array(list(grni111_hbeef_eads_dict.keys())) / 100
hbeef_energies = np.array([grni111_hbeef_eads_dict[z]['hBEEF-vdW'] for z in hbeef_zlist])


beef_spline = UnivariateSpline(zheight_angstrom, beef_energies, s=4)
hbeef_spline = UnivariateSpline(hbeef_zheight_angstrom, hbeef_energies, s=4)

xfit = np.linspace(min(zheight_angstrom), max(zheight_angstrom), 100)
yfit_beef = beef_spline(xfit)  / kjmol_to_meV
yfit_hbeef = hbeef_spline(xfit)  / kjmol_to_meV


rpa_zheight_angstrom = np.array(list(grni111_rpa_eads_dict.keys())) / 100
rpa_energies = np.array([grni111_rpa_eads_dict[z]['RPA'] for z in rpa_zlist])
dhbeef_energies = np.array([grni111_rpa_eads_dict[z]['dhBEEF-vdW'] for z in rpa_zlist])

rpa_xfit = np.linspace(min(rpa_zheight_angstrom), max(rpa_zheight_angstrom), 100)
rpa_yfit = UnivariateSpline(rpa_zheight_angstrom, rpa_energies, s=4)(rpa_xfit) / kjmol_to_meV
dhbeef_yfit = UnivariateSpline(rpa_zheight_angstrom, dhbeef_energies, s=4)(rpa_xfit)  / kjmol_to_meV

axs[1].plot(rpa_xfit, rpa_yfit, color=color_dict[method_dict['RPA']['color']], linestyle='--')
axs[1].scatter(rpa_zheight_angstrom, rpa_energies  / kjmol_to_meV, label='RPA', color=color_dict[method_dict['RPA']['color']], facecolor='none',linewidth=1.5)

axs[1].plot(rpa_xfit, dhbeef_yfit, color=color_dict[method_dict['dhBEEFvdW']['color']], linestyle='--')
axs[1].scatter(rpa_zheight_angstrom, dhbeef_energies  / kjmol_to_meV, label='dhBEEFvdW', color=color_dict[method_dict['dhBEEFvdW']['color']], facecolor='none',linewidth=1.5)

axs[1].scatter(zheight_angstrom, beef_energies  / kjmol_to_meV, label='BEEF-vdW', color=color_dict[method_dict['BEEFvdW']['color']], facecolor='none',linewidth=1.5)
axs[1].plot(xfit, yfit_beef, color=color_dict[method_dict['BEEFvdW']['color']], linestyle='--')

axs[1].scatter(hbeef_zheight_angstrom, hbeef_energies  / kjmol_to_meV, label='hBEEF-vdW', color=color_dict[method_dict['hBEEFvdW']['color']], facecolor='none',linewidth=1.5)
axs[1].plot(xfit, yfit_hbeef, color=color_dict[method_dict['hBEEFvdW']['color']], linestyle='--')



# Plot DFT XC results as points
for dft_xc in ['01_PBE', '14_r2SCANrVV10']:
    dft_zheight_angstrom = np.array(dft_xc_zlist) / 100
    dft_energies = np.array([grni111_dft_xc_eads_dict[z][dft_xc] for z in dft_xc_zlist])
    axs[1].scatter(dft_zheight_angstrom, dft_energies  / kjmol_to_meV, label=method_dict[dft_xc]['label'], color=color_dict[method_dict[dft_xc]['color']], facecolor='none', marker='o',linewidth=1.5)
    dft_xfit = np.linspace(min(dft_zheight_angstrom), max(dft_zheight_angstrom), 100)
    dft_yfit = UnivariateSpline(dft_zheight_angstrom, dft_energies, s=4)(dft_xfit)  / kjmol_to_meV  # Fit a spline to the DFT XC energies
    axs[1].plot(dft_xfit, dft_yfit, linestyle='--',color=color_dict[method_dict[dft_xc]['color']])

# Experimental bounds
x_exp_min, x_exp_max = 2.04, 2.18
y_exp_min, y_exp_max = -11.2, -7.2

# Square outline
axs[1].plot(
    [x_exp_min, x_exp_max, x_exp_max, x_exp_min, x_exp_min],
    [y_exp_min, y_exp_min, y_exp_max, y_exp_max, y_exp_min],
    color='black',
    linewidth=1.0,
    label='Experiment'
)

# Light fill inside the square
axs[1].fill(
    [x_exp_min, x_exp_max, x_exp_max, x_exp_min],
    [y_exp_min, y_exp_min, y_exp_max, y_exp_max],
    color='gray',
    alpha=0.3,
    edgecolor=None
)

# Experimental center
x0 = (x_exp_min + x_exp_max) / 2
y0 = (y_exp_min + y_exp_max) / 2

# Experimental half-widths (errors)
xerr = (x_exp_max - x_exp_min) / 2
yerr = (y_exp_max - y_exp_min) / 2

axs[1].errorbar(
    x0, y0,
    xerr=xerr,
    yerr=yerr,
    fmt='s',              # square marker
    color='black',
    ecolor='black',
    elinewidth=1.0,
    capsize=3,
    markersize=5,
    zorder=6,
    label='_nolegend_'    # legend already handled by box
)


# Only add legend label for Experiment
for handle in axs[1].get_legend_handles_labels()[0]:
    if handle.get_label() == 'Experiment':
        handle.set_label('Experiment')
    else:
        handle.set_label('_nolegend_')
axs[1].legend(fontsize=8, loc='upper right')


axs[1].set_ylim([-15,5])
axs[1].set_xlim([1.8,3.6])

axs[1].set_ylabel('Adsorption energy per C atom (kJ/mol)')
axs[1].set_xlabel('Distance between graphene and Ni(111) (Å)')


axs[0].set_xlim([-20,20])
axs[0].spines['top'].set_visible(True)
axs[0].spines['bottom'].set_visible(False)
axs[0].spines['right'].set_visible(False)

# Place the y-axis spine (and thus the ticks) at x = 0
axs[0].spines['left'].set_position(('data', 0))

# Now ticks and labels will be drawn along x=0
axs[0].yaxis.set_ticks_position('left')
axs[0].yaxis.set_label_position('left')

# Move x-ticks and labels to top
axs[0].xaxis.tick_top()
axs[0].xaxis.set_label_position('top')

axs[1].xaxis.tick_top()
axs[1].xaxis.set_label_position('top')

# Define ticks
axs[0].set_xticks([-20,-10,0,10,20])
axs[0].set_xlabel(r'Relative energy (Top --- Hollow) (kJ/mol)')


yticks = list(np.arange(num_sys)*spacing) + list(np.arange(num_sys)*spacing + 0.4) + list(np.arange(num_sys)*spacing + 0.8) + list(np.arange(num_sys)*spacing + 1.2) + list(np.arange(num_sys)*spacing + 1.6) + list(np.arange(num_sys)*spacing + 2.0)
axs[0].set_yticks(yticks)
axs[0].set_yticklabels(['']*len(yticks))

axs[0].legend(fontsize=8, loc='center right')


plt.savefig('Figures/MAIN_Figure-CO_and_Graphene.png')

# Barrier heights

## NBH56

In [118]:
nbh56_df = pd.read_csv('Data/NBH56/NBH56.csv')
nbh56_eint_dict = {row['Reaction']: {method: 0.0 for method in ['CCSD(T)','dhBEEFvdW','hBEEFvdW','RPA',
'BEEFvdW'] + dft_xc_list} for index, row in nbh56_df.iterrows()}

for index, row in nbh56_df.iterrows():
    stoic_labels = row['Stoichiometry'].split(',')[1::2]
    stoic_factors = row['Stoichiometry'].split(',')[::2]
    for dft_xc in dft_xc_list:
        total_energy = 0.0

        for sys_idx in range(len(stoic_labels)):
            energy = get_energy(f'Data/NBH56/DFT/{stoic_labels[sys_idx]}/OUTCAR_{dft_xc}.gz', code ='vasp')
            total_energy += energy * float(stoic_factors[sys_idx])
        nbh56_eint_dict[row['Reaction']][dft_xc] = total_energy * 1000 / kjmol_to_meV

    # Add MRCC RPA numbers
    mrccrpa_ene_dict = {
        'EXX_QZ': 0,
        'RPA_QZ': 0,
        'EXX_5Z': 0,
        'RPA_5Z': 0,
        'EXX_CBS': 0,
        'RPA_CBS': 0
    }

    for sys_idx in range(len(stoic_labels)):
        mrccrpa_ene_dict['EXX_QZ'] += get_energy(f'Data/NBH56/RPA/{stoic_labels[sys_idx]}/mrcc_1.out.gz', code='mrcc', method='rpa_exx') * float(stoic_factors[sys_idx])
        mrccrpa_ene_dict['RPA_QZ'] += get_energy(f'Data/NBH56/RPA/{stoic_labels[sys_idx]}/mrcc_1.out.gz', code='mrcc', method='rpa_corr') * float(stoic_factors[sys_idx])
        mrccrpa_ene_dict['EXX_5Z'] += get_energy(f'Data/NBH56/RPA/{stoic_labels[sys_idx]}/mrcc_2.out.gz', code='mrcc', method='rpa_exx') * float(stoic_factors[sys_idx])
        mrccrpa_ene_dict['RPA_5Z'] += get_energy(f'Data/NBH56/RPA/{stoic_labels[sys_idx]}/mrcc_2.out.gz', code='mrcc', method='rpa_corr') * float(stoic_factors[sys_idx])
    nbh56_eint_dict[row['Reaction']]['RPA'] = get_cbs(mrccrpa_ene_dict['EXX_QZ'], mrccrpa_ene_dict['RPA_QZ'], mrccrpa_ene_dict['EXX_5Z'], mrccrpa_ene_dict['RPA_5Z'],X=4,Y=5,family='acc',output=False)[-1] * Hartree * 1000 / kjmol_to_meV
    mrccrpa_ene_dict['EXX_CBS'] = get_cbs(mrccrpa_ene_dict['EXX_QZ'], mrccrpa_ene_dict['RPA_QZ'], mrccrpa_ene_dict['EXX_5Z'], mrccrpa_ene_dict['RPA_5Z'],X=4,Y=5,family='acc',output=False)[0] * Hartree
    mrccrpa_ene_dict['RPA_CBS'] = get_cbs(mrccrpa_ene_dict['EXX_QZ'], mrccrpa_ene_dict['RPA_QZ'], mrccrpa_ene_dict['EXX_5Z'], mrccrpa_ene_dict['RPA_5Z'],X=4,Y=5,family='acc',output=False)[1] * Hartree


    # Add hBEEFvdW and BEEFvdW
    total_energy = 0.0
    rpac_total_energy = 0.0
    rpa_exx_total_energy = 0.0
    exx_total_energy = 0.0
    nlc_total_energy = 0.0
    beefc_total_energy = 0.0
    beefx_total_energy = 0.0
    beef_total_energy = 0.0
    for sys_idx in range(len(stoic_labels)):
        beef_total_energy += get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_00_BEEF_XC_vdW.gz', code='vasp') * float(stoic_factors[sys_idx])
        nlc_total_energy += (get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_00_BEEF_XC_vdW.gz', code='vasp') - get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_01_BEEF_XC.gz', code='vasp')) * float(stoic_factors[sys_idx])
        beefc_total_energy += (get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_01_BEEF_XC.gz', code='vasp') - get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_02_BEEF_X.gz', code='vasp')) * float(stoic_factors[sys_idx])
        beefx_total_energy += get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_02_BEEF_X.gz', code='vasp') * float(stoic_factors[sys_idx])
        exx_total_energy += get_energy(f'Data/NBH56/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_03_EXX.gz', code='vasp') * float(stoic_factors[sys_idx])
    # Add BEEF-vdW
    nbh56_eint_dict[row['Reaction']]['BEEFvdW'] = beef_total_energy * 1000 / kjmol_to_meV
    # Add hBEEF-vdW
    frac = 0.175
    hbeef_total_energy = frac * exx_total_energy + (1-frac)*beefx_total_energy + beefc_total_energy + nlc_total_energy
    nbh56_eint_dict[row['Reaction']]['hBEEFvdW'] = hbeef_total_energy * 1000 / kjmol_to_meV
    # Add dhBEEF-vdW
    x_frac = 0.25
    c_frac = 0.15
    dhbeef_total_energy = x_frac * exx_total_energy + (1-x_frac)*beefx_total_energy + c_frac*(mrccrpa_ene_dict['RPA_CBS']) + (1-c_frac)*beefc_total_energy + nlc_total_energy

    nbh56_eint_dict[row['Reaction']]['dhBEEFvdW'] = dhbeef_total_energy * 1000 / kjmol_to_meV

    nbh56_eint_dict[row['Reaction']]['CCSD(T)'] = row['Reference'] * Hartree * 1000 / kjmol_to_meV

# Calculate MAD for each method
nbh56_eint_df = pd.DataFrame.from_dict(nbh56_eint_dict, orient='index')
mad_dict = {}
for method in method_dict.keys():
    mad_dict[method] = np.mean(np.abs(nbh56_eint_df[method] - nbh56_eint_df['CCSD(T)']))
nbh56_eint_df.loc['MAD'] = mad_dict


# Break up into two DataFrames for better display
nbh56_eint_df_1 = nbh56_eint_df[['CCSD(T)', 'hBEEFvdW', 'dhBEEFvdW', 'RPA', 'BEEFvdW']]
nbh56_eint_df_2 = nbh56_eint_df[dft_xc_list]

nbh56_eint_df_1.columns = [
    method_dict[method]['label'] if method != 'CCSD(T)' else method
    for method in nbh56_eint_df_1.columns.tolist()
]

nbh56_eint_df_2.columns = [method_dict[method]['label'] for method in nbh56_eint_df_2.columns.tolist()]

convert_df_to_latex_input(
    df=nbh56_eint_df_1.round(1),
    start_input = '\\begin{table}[h]\n',
    label = "tab:nbh56_1",
    caption = r"Reaction energies of the NBH56 benchmark set calculated with hBEEF-vdW, dhBEEF-vdW, RPA@PBE, BEEF-vdW. These are compared to CCSD(T) estimates from Gold-Standard Chemical Database 137~\cite{liangGoldstandardChemicalDatabase2025} (GSCDB137).  All values are in kJ/mol.",
    float_fmt = '%.1f',
    rotate_column_header=True,
    longtable=True,
    replace_input= {
        r' & \rotatebox{90}{CCSD(T)}': r'Label & \rotatebox{90}{CCSD(T)}',
        r'76_': r'76\_',
        r'24_': r'24\_',
        r'NaN': r'-'
    },
    filename = "Tables/SI_Table-NBH56_Barrier_Heights_NSC.tex"
)

convert_df_to_latex_input(
    df=nbh56_eint_df_2.round(1),
    start_input = '\\begin{table}[h]\n',
    label = "tab:nbh56_2",
    caption = r"Reaction energies of the NBH56 benchmark set calculated with various DFT functionals. All values are in kJ/mol.",
    float_fmt = '%.1f',
    rotate_column_header=True,
    longtable=True,
    replace_input= {
        r' & \rotatebox{90}{PBE}': r'Label & \rotatebox{90}{PBE}',
        r'76_': r'76\_',
        r'24_': r'24\_',
        r'NaN': r'-'
    },
    filename = "Tables/SI_Table-NBH56_Barrier_Heights_DFT.tex"
)


nbh56_eint_df.columns = [
    method_dict[method]['label'] if method != 'CCSD(T)' else method
    for method in nbh56_eint_df.columns.tolist()
]

display(nbh56_eint_df.round(1))

,CCSD(T),dhBEEF-vdW,hBEEF-vdW,RPA@PBE,BEEF-vdW,PBE,PBEsol,RPBE,revPBE,BLYP,r$^2$SCAN,MS2,revTPSS,PBE-D3,PBE-DdsC,RPBE-D3,optPBE-vdW,rev-vdW-DF2,r$^2$SCAN+rVV10
BH76_3,176.1,129.0,132.4,161.8,117.3,105.1,91.1,114.9,114.6,100.8,112.4,120.4,108.7,104.9,104.8,112.5,101.3,95.4,111.7
BH76_7,127.6,85.2,89.1,115.6,71.1,70.1,64.1,74.3,75.1,59.9,71.6,85.4,69.1,68.7,68.9,71.8,59.1,59.3,70.9
BH76_8,238.1,195.6,204.5,223.5,185.1,164.1,148.9,182.1,181.8,168.6,188.4,166.2,162.5,162.0,162.1,175.0,167.2,160.4,186.4
BH76_9,6.3,-30.9,-28.4,1.4,-43.7,-44.5,-51.3,-37.3,-37.0,-52.7,-44.3,-38.2,-46.6,-44.9,-44.7,-39.9,-51.9,-53.6,-44.9
BH76_10,438.5,391.0,406.6,430.6,358.6,326.3,308.5,347.6,346.5,332.2,367.0,342.7,330.9,325.8,326.0,344.0,336.0,325.7,365.3
BH76_31,13.4,1.2,1.0,18.5,-3.9,-7.8,-16.9,-2.3,-2.7,-9.0,-14.3,-3.5,-21.5,-8.6,-8.4,-9.6,-13.8,-15.8,-14.8
BH76_32,95.4,107.5,109.6,98.8,106.9,105.6,108.4,104.1,104.4,100.0,102.3,110.9,92.7,105.0,105.2,100.8,105.2,105.6,102.3
BH76_35,26.8,17.5,21.4,33.0,16.8,6.9,-5.9,19.8,19.8,19.8,9.3,9.4,13.0,2.0,0.5,-0.7,5.7,2.5,6.8
BH76_36,138.1,128.2,131.4,136.3,118.1,124.8,134.3,119.0,120.1,103.4,129.6,134.6,117.7,123.9,125.0,120.6,114.7,120.6,130.1
BH76_39,25.5,-1.7,-1.7,15.0,-8.0,2.8,-0.1,3.5,4.1,-9.7,-2.9,-3.2,-20.9,2.4,2.7,-1.5,-16.7,-12.9,-3.3


## S66

In [119]:
# Extract all S66 entries from the dataframe
s66_df = pd.read_csv('Data/S66/S66.csv')

s66_eint_dict = {row['Reaction']: {dft_xc: 0.0 for dft_xc in ['CCSD(T)'] + ['RPA', 'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW'] + dft_xc_list} for index, row in s66_df.iterrows()}

# Iterate over rows of S66 dataframe to extract system names
for index, row in s66_df.iterrows():
    stoic_labels = row['Stoichiometry'].split(',')[1::2]
    stoic_factors = row['Stoichiometry'].split(',')[::2]

    for dft_xc in dft_xc_list:
        total_energy = 0.0

        for sys_idx in range(len(stoic_labels)):
            energy = get_energy(f'Data/S66/DFT/{stoic_labels[sys_idx]}/OUTCAR_{dft_xc}.gz', code='vasp')
            total_energy += energy * float(stoic_factors[sys_idx])
        s66_eint_dict[row['Reaction']][dft_xc] = total_energy * 1000 / kjmol_to_meV

    # Add MRCC RPA numbers
    mrccrpa_ene_dict = {
        'EXX_QZ': 0,
        'RPA_QZ': 0,
        'EXX_5Z': 0,
        'RPA_5Z': 0,
        'EXX_CBS': 0,
        'RPA_CBS': 0
    }
    for sys_idx in range(len(stoic_labels)):
        mrccrpa_ene_dict['EXX_QZ'] += get_energy(f'Data/S66/RPA/{stoic_labels[sys_idx]}/mrcc_1.out.gz', code='mrcc', method='rpa_exx') * float(stoic_factors[sys_idx])
        mrccrpa_ene_dict['RPA_QZ'] += get_energy(f'Data/S66/RPA/{stoic_labels[sys_idx]}/mrcc_1.out.gz', code='mrcc', method='rpa_corr') * float(stoic_factors[sys_idx])
        mrccrpa_ene_dict['EXX_5Z'] += get_energy(f'Data/S66/RPA/{stoic_labels[sys_idx]}/mrcc_2.out.gz', code='mrcc', method='rpa_exx') * float(stoic_factors[sys_idx])
        mrccrpa_ene_dict['RPA_5Z'] += get_energy(f'Data/S66/RPA/{stoic_labels[sys_idx]}/mrcc_2.out.gz', code='mrcc', method='rpa_corr') * float(stoic_factors[sys_idx])

    s66_eint_dict[row['Reaction']]['RPA'] = get_cbs(mrccrpa_ene_dict['EXX_QZ'], mrccrpa_ene_dict['RPA_QZ'], mrccrpa_ene_dict['EXX_5Z'], mrccrpa_ene_dict['RPA_5Z'],X=4,Y=5,family='acc',output=False)[-1]*Hartree*1000 / kjmol_to_meV
    mrccrpa_ene_dict['EXX_CBS'] = get_cbs(mrccrpa_ene_dict['EXX_QZ'], mrccrpa_ene_dict['RPA_QZ'], mrccrpa_ene_dict['EXX_5Z'], mrccrpa_ene_dict['RPA_5Z'],X=4, Y=5, family='acc', output=False)[-1] * Hartree
    mrccrpa_ene_dict['RPA_CBS'] = get_cbs(mrccrpa_ene_dict['EXX_QZ'], mrccrpa_ene_dict['RPA_QZ'], mrccrpa_ene_dict['EXX_5Z'], mrccrpa_ene_dict['RPA_5Z'], X=4, Y=5, family='acc', output=False)[-1] * Hartree

    # Add hBEEFvdW and BEEFvdW
    rpac_total_energy = 0.0
    rpa_exx_total_energy = 0.0
    exx_total_energy = 0.0
    nlc_total_energy = 0.0
    beefc_total_energy = 0.0
    beefx_total_energy = 0.0
    beef_total_energy = 0.0
    for sys_idx in range(len(stoic_labels)):
        beef_total_energy += get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_00_BEEF_XC_vdW.gz', code='vasp') * float(stoic_factors[sys_idx])
        nlc_total_energy += (get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_00_BEEF_XC_vdW.gz', code='vasp') - get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_01_BEEF_XC.gz', code='vasp')) * float(stoic_factors[sys_idx])
        beefc_total_energy += (get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_01_BEEF_XC.gz', code='vasp') - get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_02_BEEF_X.gz', code='vasp')) * float(stoic_factors[sys_idx])
        beefx_total_energy += get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_02_BEEF_X.gz', code='vasp') * float(stoic_factors[sys_idx])
        exx_total_energy += get_energy(f'Data/S66/hBEEFvdW/{stoic_labels[sys_idx]}/OUTCAR_03_EXX.gz', code='vasp') * float(stoic_factors[sys_idx])

    # Add BEEF-vdW
    s66_eint_dict[row['Reaction']]['BEEFvdW'] = beef_total_energy * 1000 / kjmol_to_meV
    # Add hBEEF-vdW
    frac = 0.175
    hbeef_total_energy = frac * exx_total_energy + (1-frac)*beefx_total_energy + beefc_total_energy + nlc_total_energy
    s66_eint_dict[row['Reaction']]['hBEEFvdW'] = hbeef_total_energy * 1000 / kjmol_to_meV
    # Add dhBEEF-vdW
    x_frac = 0.25
    c_frac = 0.15

    dhbeef_total_energy = x_frac * exx_total_energy + (1-x_frac)*beefx_total_energy + c_frac*(mrccrpa_ene_dict['RPA_CBS']) + (1-c_frac)*beefc_total_energy + nlc_total_energy
    s66_eint_dict[row['Reaction']]['dhBEEFvdW'] = dhbeef_total_energy * 1000 / kjmol_to_meV

    s66_eint_dict[row['Reaction']]['CCSD(T)'] = row['Reference'] * Hartree * 1000 / kjmol_to_meV

# Calculate MAD for each method
s66_eint_df = pd.DataFrame.from_dict(s66_eint_dict, orient='index')
mad_dict = {}
for method in method_dict.keys():
    mad_dict[method] = np.mean(np.abs(s66_eint_df[method] - s66_eint_df['CCSD(T)']))
s66_eint_df.loc['MAD'] = mad_dict

# Break up into two DataFrames for better display
s66_eint_df_1 = s66_eint_df[['CCSD(T)', 'hBEEFvdW', 'dhBEEFvdW', 'RPA', 'BEEFvdW']]
s66_eint_df_2 = s66_eint_df[dft_xc_list]

s66_eint_df_1.columns = [
    method_dict[method]['label'] if method != 'CCSD(T)' else method
    for method in s66_eint_df_1.columns.tolist()
]

s66_eint_df_2.columns = [method_dict[method]['label'] for method in s66_eint_df_2.columns.tolist()]
convert_df_to_latex_input(
    df=s66_eint_df_1.round(1),
    start_input = '\\begin{table}[h]\n',
    label = "tab:s66_1",
    caption = r"Interaction energies of the S66 benchmark set calculated with hBEEF-vdW, dhBEEF-vdW, RPA@PBE, BEEF-vdW. These are compared to CCSD(T) estimates from Gold-Standard Chemical Database 137~\cite{liangGoldstandardChemicalDatabase2025} (GSCDB137).  All values are in kJ/mol.",
    float_fmt = '%.1f',
    rotate_column_header=True,
    longtable=True,
    replace_input= {
        r' & \rotatebox{90}{CCSD(T)}': r'Label & \rotatebox{90}{CCSD(T)}',
        r'66_': r'66\_',
        r'NaN': r'-'
    },
    filename = "Tables/SI_Table-S66_Interaction_Energies_NSC.tex"
)

convert_df_to_latex_input(
    df=s66_eint_df_2.round(1),
    start_input = '\\begin{table}[h]\n',
    label = "tab:s66_2",
    caption = r"Interaction energies of the S66 benchmark set calculated with various DFT functionals. All values are in kJ/mol.",
    float_fmt = '%.1f',
    rotate_column_header=True,
    longtable=True,
    replace_input= {
        r' & \rotatebox{90}{PBE}': r'Label & \rotatebox{90}{PBE}',
        r'66_': r'66\_',
        r'NaN': r'-'
    },
    filename = "Tables/SI_Table-S66_Interaction_Energies_DFT.tex"
)

s66_eint_df.columns = [
    method_dict[method]['label'] if method != 'CCSD(T)' else method
    for method in s66_eint_df.columns.tolist()
]
display(s66_eint_df.round(1))


,CCSD(T),RPA@PBE,BEEF-vdW,hBEEF-vdW,dhBEEF-vdW,PBE,PBEsol,RPBE,revPBE,BLYP,r$^2$SCAN,MS2,revTPSS,PBE-D3,PBE-DdsC,RPBE-D3,optPBE-vdW,rev-vdW-DF2,r$^2$SCAN+rVV10
S66_1,-20.8,-18.4,-18.7,-19.8,-22.1,-21.1,-24.3,-15.5,-14.6,-17.8,-22.4,-22.3,-18.8,-22.9,-22.7,-20.2,-20.7,-21.0,-23.4
S66_2,-23.7,-21.0,-21.0,-22.3,-24.9,-22.2,-26.1,-15.6,-14.6,-18.4,-24.8,-24.0,-19.5,-25.4,-25.7,-22.8,-24.0,-23.8,-26.3
S66_3,-29.2,-25.9,-27.3,-28.1,-31.0,-30.7,-35.3,-23.5,-23.0,-25.9,-31.3,-30.2,-27.5,-33.9,-33.8,-31.0,-30.5,-31.1,-32.8
S66_4,-34.3,-30.7,-30.0,-32.1,-36.1,-31.3,-36.7,-22.2,-21.2,-25.9,-35.9,-34.6,-28.1,-35.8,-36.5,-33.1,-34.2,-34.0,-38.2
S66_5,-24.3,-21.6,-22.0,-23.3,-26.0,-21.9,-25.9,-15.2,-14.2,-17.7,-24.7,-23.7,-19.1,-26.1,-26.6,-23.5,-25.6,-24.8,-26.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S66_63,-15.6,-12.9,-12.0,-14.0,-15.4,-2.3,-6.4,5.8,7.9,6.8,-10.2,-7.8,-0.4,-15.1,-16.1,-15.0,-18.4,-13.7,-16.1
S66_64,-12.4,-10.1,-11.0,-12.4,-13.4,-4.1,-6.9,2.3,4.5,2.8,-8.5,-7.1,-1.5,-13.3,-14.0,-12.5,-14.8,-11.1,-12.3
S66_65,-17.1,-15.1,-15.9,-16.7,-18.6,-15.5,-17.8,-11.3,-10.3,-11.9,-15.1,-15.3,-14.1,-19.0,-18.7,-18.2,-18.0,-16.9,-16.8
S66_66,-16.5,-13.5,-13.4,-14.6,-16.0,-7.4,-11.3,0.3,2.2,-0.2,-12.5,-10.2,-5.0,-16.9,-17.8,-16.5,-18.6,-15.5,-16.8


## SE20

In [120]:
# Calculate the surface energies for the se20 dataset

se20_df = pd.read_csv('Data/SE20/SE20.csv', index_col='sysnames')
# Create a dictionary to hold the surface energies
se20_method_dict = {surface: {'Experiment':0,'hBEEFvdW': 0, 'dhBEEFvdW': 0, 'RPA': 0, 'BEEFvdW': 0} | {dft_xc:0 for dft_xc in dft_xc_list} for surface in se20_df.index}
for surface in se20_df.index:
    element = surface[:-3]
    cell_type = se20_df.loc[surface, 'cell_type']
    bulk_system = element + f'_{cell_type}'
    surface_label = surface[:-3] + f'_{surface[-3:]}'

    for dft_xc in dft_xc_list:
        bulk_struct = read(f'Data/SE20/Bulk/DFT/{element}/OUTCAR_{dft_xc}.gz')
        surface_struct = read(f'Data/SE20/Surface/DFT/{surface_label}/OUTCAR_{dft_xc}.gz')
        bulk_energy = bulk_struct.calc.results['energy']
        surface_energy = surface_struct.calc.results['energy']
        num_atoms_bulk = len(bulk_struct)
        num_atoms_surf = len(surface_struct)
        # Calculate the surface area in meter squared
        surface_area = surface_struct.get_volume() / surface_struct.get_cell()[2,2] * 1e-20  # m^2
        surf_ene = 0.5 * 1.60218e-19 * (surface_energy - (num_atoms_surf / num_atoms_bulk) * bulk_energy) / surface_area
        se20_method_dict[surface][dft_xc] = surf_ene
    se20_method_dict[surface]['Experiment'] = se20_df.loc[surface, 'Exp_SE']  # J/m^2
    # BEEF-vdW and dh BEEF-vdW
    beef_energy_dict = {method: {'Bulk': 0.0, 'Surface': 0.0} for method in ['rpac', 'rpa_exx', 'exx', 'nlc', 'beefc', 'beefx', 'beef']}
    # Start with Bulk energy
    for system in ['Bulk', 'Surface']:
        if system == 'Bulk':
            root_dir = f'Data/SE20/Bulk'
            syslabel = element
        else:
            root_dir = 'Data/SE20/Surface'
            syslabel = surface_label
        beef_energy_dict['beef'] [system] = get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_00_BEEF_XC_vdW.gz', code='vasp')
        beef_energy_dict['nlc'] [system] = (get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_00_BEEF_XC_vdW.gz', code='vasp') - get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_01_BEEF_XC.gz', code='vasp'))
        beef_energy_dict['beefc'] [system] = (get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_01_BEEF_XC.gz', code='vasp') - get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_02_BEEF_X.gz', code='vasp'))
        beef_energy_dict['beefx'] [system] = get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_02_BEEF_X.gz', code='vasp')
        beef_energy_dict['exx'] [system] = get_energy(f'{root_dir}/hBEEFvdW/{syslabel}/OUTCAR_03_EXX.gz', code='vasp')
        beef_energy_dict['rpa_exx'] [system] = get_energy(f'{root_dir}/RPA/{syslabel}/OUTCAR_02_EXX.gz', code='vasp')
        beef_energy_dict['rpac'] [system] = get_energy(f'{root_dir}/RPA/{syslabel}/OUTCAR_04_RPA.gz', code='vasp_rpa')
    # Now calculate surface energy contributions
    rpac_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['rpac']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['rpac']['Bulk']) / surface_area
    rpa_exx_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['rpa_exx']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['rpa_exx']['Bulk']) / surface_area
    exx_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['exx']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['exx']['Bulk']) / surface_area
    nlc_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['nlc']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['nlc']['Bulk']) / surface_area
    beefc_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['beefc']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['beefc']['Bulk']) / surface_area
    beefx_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['beefx']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['beefx']['Bulk']) / surface_area
    beef_total_energy = 0.5 * 1.60218e-19 * (beef_energy_dict['beef']['Surface'] - (num_atoms_surf / num_atoms_bulk) * beef_energy_dict['beef']['Bulk']) / surface_area

    # Add BEEF-vdW
    se20_method_dict[surface]['BEEFvdW'] = beef_total_energy
    # Add hBEEF-vdW
    frac = 0.175
    hbeef_total_energy = frac * exx_total_energy + (1-frac)*beefx_total_energy + beefc_total_energy + nlc_total_energy
    se20_method_dict[surface]['hBEEFvdW'] = hbeef_total_energy
    # Add dhBEEF-vdW
    x_frac = 0.25
    c_frac = 0.15
    dhbeef_total_energy = x_frac * exx_total_energy + (1-x_frac)*beefx_total_energy + c_frac*(rpac_total_energy) + (1-c_frac)*beefc_total_energy + nlc_total_energy
    se20_method_dict[surface]['dhBEEFvdW'] = dhbeef_total_energy
    # Add RPA
    rpa_total_energy = rpa_exx_total_energy + rpac_total_energy
    se20_method_dict[surface]['RPA'] = rpa_total_energy




In [121]:
# Calculate MAD for each method
se20_eint_df = pd.DataFrame.from_dict(se20_method_dict, orient='index')
mad_dict = {}
for method in se20_method_dict['Li110'].keys():
    mad_dict[method] = np.mean(np.abs(se20_eint_df[method] - se20_eint_df['Experiment']))

# Change e.g., Li110 to Li(110)
se20_eint_df.index = [f'{system[:-3]}({system[-3:]})' for system in se20_eint_df.index.tolist()]

se20_eint_df.loc['MAD'] = mad_dict

# Break up into two DataFrames for better display
se20_eint_df_1 = se20_eint_df[['Experiment', 'hBEEFvdW', 'dhBEEFvdW', 'RPA', 'BEEFvdW']]
se20_eint_df_2 = se20_eint_df[dft_xc_list]

se20_eint_df_1.columns = [
    method_dict[method]['label'] if method != 'Experiment' else method
    for method in se20_eint_df_1.columns.tolist()
]

se20_eint_df_2.columns = [method_dict[method]['label'] for method in se20_eint_df_2.columns.tolist()]

convert_df_to_latex_input(
    df=se20_eint_df_1.round(2),
    start_input = '\\begin{table}[h]\n',
    label = "tab:se20_1",
    caption = r"Surface energies of the SE20 benchmark set calculated with hBEEF-vdW, dhBEEF-vdW, RPA@PBE, BEEF-vdW. These are compared to experimental estimates from Lundgaard~\etal{}~\cite{lundgaardMBEEFvdWRobustFitting2016}. All values are in J/m$^2$.",
    float_fmt = '%.2f',
    rotate_column_header=True,
    replace_input= {
        r' & \rotatebox{90}{Experiment}': r'Surface & \rotatebox{90}{Experiment}',
        r'NaN': r'-'
    },
    filename = "Tables/SI_Table-SE20_Surface_Energies_NSC.tex"
)

convert_df_to_latex_input(
    df=se20_eint_df_2.round(2),
    start_input = '\\begin{table}[h]\n',
    label = "tab:se20_2",
    caption = r"Surface energies of the SE20 benchmark set calculated with various DFT functionals. These are compared to experimental estimates from Lundgaard~\etal{}~\cite{lundgaardMBEEFvdWRobustFitting2016}. All values are in J/m$^2$.",
    float_fmt = '%.2f',
    rotate_column_header=True,
    replace_input= {
        r' & \rotatebox{90}{PBE}': r'Surface & \rotatebox{90}{PBE}',
        r'NaN': r'-'
    },
    filename = "Tables/SI_Table-SE20_Surface_Energies_DFT.tex"
)

# Change column names to labels
se20_eint_df.columns = [method_dict[method]['label'] if method in method_dict else method for method in se20_eint_df.columns.tolist()]

display(se20_eint_df.round(2))

,Experiment,hBEEF-vdW,dhBEEF-vdW,RPA@PBE,BEEF-vdW,PBE,PBEsol,RPBE,revPBE,BLYP,r$^2$SCAN,MS2,revTPSS,PBE-D3,PBE-DdsC,RPBE-D3,optPBE-vdW,rev-vdW-DF2,r$^2$SCAN+rVV10
Li(110),0.52,0.55,0.56,0.49,0.56,0.49,0.52,0.46,0.46,0.34,0.51,0.47,0.52,0.60,0.52,0.88,0.52,0.55,0.55
Na(110),0.26,0.25,0.26,0.22,0.24,0.21,0.23,0.19,0.19,0.11,0.24,0.24,0.24,0.30,0.25,0.44,0.24,0.25,0.27
K(110),0.14,0.12,0.13,0.14,0.12,0.11,0.12,0.08,0.08,0.04,0.12,0.12,0.12,0.15,0.13,0.20,0.13,0.13,0.14
Rb(110),0.11,0.10,0.11,0.11,0.09,0.08,0.10,0.06,0.06,0.03,0.09,0.09,0.09,0.12,0.11,0.17,0.11,0.11,0.11
Ba(110),0.38,0.36,0.39,0.41,0.36,0.31,0.36,0.26,0.27,0.20,0.34,0.35,0.36,0.37,0.49,0.64,0.38,0.40,0.40
Ca(111),0.50,0.52,0.53,0.52,0.52,0.47,0.51,0.43,0.43,0.31,0.50,0.51,0.53,0.55,0.51,0.79,0.52,0.54,0.56
Sr(111),0.41,0.39,0.41,0.41,0.38,0.34,0.39,0.30,0.31,0.21,0.38,0.41,0.40,0.41,0.39,0.64,0.39,0.41,0.43
Nb(110),2.68,2.35,2.40,2.48,2.17,2.08,2.40,1.88,1.93,1.44,2.21,2.42,2.52,2.82,2.34,5.21,2.17,2.40,2.49
Ta(110),3.03,2.76,2.80,2.79,2.56,2.37,2.67,2.21,2.25,1.67,2.62,2.76,2.83,3.16,2.68,5.58,2.45,2.69,2.89
Mo(110),2.95,3.33,3.27,3.50,3.01,2.89,3.28,2.68,2.73,2.11,3.11,3.34,3.41,3.68,3.24,5.94,2.98,3.27,3.44


## SBH17

In [122]:
# Pandas load the SBH17 dataset

sbh17_df = pd.read_csv('Data/SBH17/SBH17.csv')
sbh17_df.index = ['H2Cu111', 'H2Cu100', 'H2Cu110', 'H2Pt111', 'H2Pt211', 'H2Ru0001', 'H2Ni111', 'H2Ag111',
                  'N2Ru0001', 'N2Ru1010', 'CH4Ni111', 'CH4Ni100', 'CH4Ni211', 'CH4Pt111', 'CH4Pt211',
                  'CH4Ir111', 'CH4Ru0001']

w_ea_dict = {
    'N2Ru1010': 6.582553276239793,
    'N2Ru0001': 7.383190599482175,
    'H2Ag111': 7.685919139613622,
    'H2Cu110': 7.713802031467835,
    'H2Cu100': 7.8842859988050185,
    'H2Cu111': 8.054371639115715,
    'H2Ni111': 8.394941246763594,
    'H2Ru0001': 8.556263692491536,
    'H2Pt211': 9.0,
    'H2Pt111': 9.066122286397132,
    'CH4Ni211': 10.721171081457877,
    'CH4Ni100': 10.92033459470225,
    'CH4Ni111': 10.992033459470225,
    'CH4Ru0001': 11.151364270065724,
    'CH4Pt211': 11.390360485958972,
    'CH4Ir111': 11.53176658036248,
    'CH4Pt111': 11.66122286397132
}

# Add a column for the W-EA term
sbh17_df['W-EA'] = sbh17_df.index.map(w_ea_dict)

# Add a column for the ordering of W-EA in ascending order
sbh17_df['W-EA Order'] = sbh17_df['W-EA'].rank()

# Get the H2, N2, CH4 list of systems based on rank of W-EA
h2_list = sbh17_df[sbh17_df.index.str.startswith('H2')].sort_values('W-EA Order').index.tolist()
n2_list = sbh17_df[sbh17_df.index.str.startswith('N2')].sort_values('W-EA Order').index.tolist()
ch4_list = sbh17_df[sbh17_df.index.str.startswith('CH4')].sort_values('W-EA Order').index.tolist()

# Make ch4_list pretty by encasing CH4Ni211 as CH4 on Ni(211)
ch4_list_pretty = [f"{x[3:5]}({x[5:]})" for x in ch4_list]
n2_list_pretty = [f"{x[2:4]}({x[4:]})" for x in n2_list]
h2_list_pretty = [f"{x[2:4]}({x[4:]})" for x in h2_list]

display(sbh17_df)

,Ns,system,functional,site,rb,Zb,theta,phi/beta,Eb,W-EA,W-EA Order
H2Cu111,1,H2 + Cu(111),SRP43,brg,1.030,1.1600,90.0,90.0,0.628,8.054372,6.0
H2Cu100,2,H2 + Cu(100),SRP43,brg,1.230,1.0054,90.0,90.0,0.740,7.884286,5.0
H2Cu110,3,H2 + Cu(110),optPBE-vdW-DF1,short-brg,1.200,0.8900,64.0,90.0,0.789,7.713802,4.0
H2Pt111,4,H2 + Pt(111),PBEa57-vdW-DF2,top (early),0.769,2.2020,90.0,0.0,-0.008,9.066122,10.0
H2Pt211,5,H2 + Pt(211),PBEa57-vdW-DF2,top,0.750,2.7900,90.0,90.0,-0.083,9.000000,9.0
H2Ru0001,6,H2 + Ru(0001),PBE-vdW-DF2,top (early),0.751,2.6050,90.0,0.0,0.004,8.556264,8.0
H2Ni111,7,H2 + Ni(111),PBE-vdW-DF2,top (early),0.763,2.0830,90.0,0.0,0.024,8.394941,7.0
H2Ag111,8,H2 + Ag(111),MS-PBEi+VV10,brg,1.224,1.1570,90.0,0.0,1.082,7.685919,3.0
N2Ru0001,9,N2 + Ru(0001),RPBE,brg,1.741,1.3180,84.0,30.0,1.840,7.383191,2.0
N2Ru1010,10,N2 + Ru(10-10),experiment,NaN,NaN,NaN,NaN,NaN,0.400,6.582553,1.0


In [123]:
sbh17_systems = "H2Cu111 H2Cu100 H2Cu110 H2Pt111 H2Pt211 H2Ru0001 H2Ni111 H2Ag111 N2Ru0001 N2Ru1010 CH4Ni111 CH4Ni100 CH4Ni211 CH4Pt111 CH4Pt211  CH4Ir111 CH4Ru0001".split()

sbh17_method_dict = {system: {'Experiment':0, 'dhBEEFvdW': 0, 'hBEEFvdW': 0, 'RPA': 0, 'BEEFvdW': 0 } | {dft_xc:0 for dft_xc in dft_xc_list} for system in sbh17_systems}

delta_layer_ene_list = []

beef_mad_list = []
beef10_mad_list = []

for system in sbh17_systems:

    # Do DFT XC functionals
    for dft_xc in dft_xc_list:
        gp_energy = get_energy(f'Data/SBH17/DFT/{system}/GP/OUTCAR_{dft_xc}.gz', code='vasp')
        ts_energy = get_energy(f'Data/SBH17/DFT/{system}/TS/OUTCAR_{dft_xc}.gz', code='vasp')
        dft_xc_ene = ts_energy - gp_energy
        sbh17_method_dict[system][dft_xc] = dft_xc_ene * 1000 / kjmol_to_meV

    # Do RPA
    ad_slab_exx_energy = get_energy(f'Data/SBH17/RPA/{system}/AD_SLAB/OUTCAR_02_EXX.gz', code='vasp')
    ad_slab_rpa_energy = get_energy(f'Data/SBH17/RPA/{system}/AD_SLAB/OUTCAR_04_RPA.gz', code='vasp_rpa')
    slab_exx_energy = get_energy(f'Data/SBH17/RPA/{system}/SLAB/OUTCAR_02_EXX.gz', code='vasp')
    slab_rpa_energy = get_energy(f'Data/SBH17/RPA/{system}/SLAB/OUTCAR_04_RPA.gz', code='vasp_rpa')
    ad_exx_energy = get_energy(f'Data/SBH17/RPA/{system}/AD/OUTCAR_02_EXX.gz', code='vasp')
    ad_rpa_energy = get_energy(f'Data/SBH17/RPA/{system}/AD/OUTCAR_04_RPA.gz', code='vasp_rpa')
    rpa_ene = (ad_slab_exx_energy - slab_exx_energy - ad_exx_energy) + (ad_slab_rpa_energy - slab_rpa_energy - ad_rpa_energy)
    sbh17_method_dict[system]['Experiment'] = float(sbh17_df.loc[system, 'Eb']) * 1000 / kjmol_to_meV
    sbh17_method_dict[system]['RPA'] = rpa_ene * 1000 / kjmol_to_meV
    # Do hBEEFvdW
    hybrid_root_folder = f'Data/SBH17/hBEEFvdW/{system}'
    calc_energy_dict = {
        '00': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '01': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '02': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '03_3': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0}
    }        

    for calc_type in calc_energy_dict:
        calc_energy_dict[calc_type]['total_product_energy'] = get_energy(f'{hybrid_root_folder}/AD_SLAB/OUTCAR_{calc_type}.gz', code='vasp')
        calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{hybrid_root_folder}/AD/OUTCAR_{calc_type}.gz', code='vasp')
        calc_energy_dict[calc_type]['reactant_surface_energy'] = get_energy(f'{hybrid_root_folder}/SLAB/OUTCAR_{calc_type}.gz', code='vasp')
        calc_energy_dict[calc_type]['eint'] = (calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV

    beefx_eint = calc_energy_dict['02']['eint']
    exx_eint = calc_energy_dict['03_3']['eint']
    beefc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['02']['eint']

    frac = 0.175
    sbh17_method_dict[system]['BEEFvdW'] = calc_energy_dict['00']['eint']
    sbh17_method_dict[system]['hBEEFvdW'] = frac * exx_eint + (1-frac)*beefx_eint + beefc_eint

    # Do dhBEEFvdW
    rpac_eint = (ad_slab_rpa_energy - slab_rpa_energy - ad_rpa_energy) * 1000 / kjmol_to_meV

    beefx_eint = calc_energy_dict['02']['eint']
    exx_eint = calc_energy_dict['03_3']['eint']
    nlc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['01']['eint']
    beefc_eint = calc_energy_dict['01']['eint'] - calc_energy_dict['02']['eint']
    x_frac = 0.25
    c_frac = 0.15
    sbh17_method_dict[system]['dhBEEFvdW'] = x_frac * exx_eint + (1-x_frac)*beefx_eint + c_frac*rpac_eint + (1-c_frac)*beefc_eint + nlc_eint


# Convert to DataFrame
sbh17_method_df = pd.DataFrame.from_dict(sbh17_method_dict, orient='index')

# Add an additional row at the bottom that is the MAD for each method
sbh17_method_df.loc['MAD (Total)'] = [np.mean(abs(sbh17_method_df[method] - sbh17_method_df['Experiment'])) for method in sbh17_method_dict['H2Cu111'].keys()]

# add an additional row for MAE on the H2 systems
h2_systems = [system for system in sbh17_systems if system.startswith('H2')]
sbh17_method_df.loc['MAD (H2)'] = [np.mean(abs(sbh17_method_df.loc[h2_systems, method] - sbh17_method_df.loc[h2_systems, 'Experiment'])) for method in sbh17_method_dict['H2Cu111'].keys()]


# Add an additional row for MAE on the CH4 systems
ch4_systems = [system for system in sbh17_systems if system.startswith('CH4')]
sbh17_method_df.loc['MAD (CH4)'] = [np.mean(abs(sbh17_method_df.loc[ch4_systems, method] - sbh17_method_df.loc[ch4_systems, 'Experiment'])) for method in sbh17_method_dict['H2Cu111'].keys()]

# Add an additional row for MAE on the N2 systems
n2_systems = [system for system in sbh17_systems if system.startswith('N2')]
sbh17_method_df.loc['MAD (N2)'] = [np.mean(abs(sbh17_method_df.loc[n2_systems, method] - sbh17_method_df.loc[n2_systems, 'Experiment'])) for method in sbh17_method_dict['H2Cu111'].keys()]

sbh17_method_df = sbh17_method_df.round(1)

# Change index
sbh17_method_df.index = [r'\ce{' + sbh17_df.loc[system, 'system'] + r'}' if system in sbh17_df.index else system for system in sbh17_method_df.index ]

# Make two separate DataFrames for DFT and non-DFT methods for better display
sbh17_method_df_1 = sbh17_method_df[['Experiment', 'hBEEFvdW', 'dhBEEFvdW', 'RPA', 'BEEFvdW']]
sbh17_method_df_2 = sbh17_method_df[dft_xc_list]

sbh17_method_df_1.columns = [
    method_dict[method]['label'] if method in method_dict else method
    for method in sbh17_method_df_1.columns.tolist()
]

sbh17_method_df_2.columns = [method_dict[method]['label'] if method in method_dict else method for method in sbh17_method_df_2.columns.tolist()]

convert_df_to_latex_input(
    df=sbh17_method_df_1.round(1),
    start_input = '\\begin{table}[h]\n',
    label = "tab:sbh17_1",
    caption = r"Barrier heights of the SBH17 benchmark set calculated with hBEEF-vdW@BEEF-vdW, dhBEEF-vdW@BEEF-vdW, RPA@PBE, BEEF-vdW. These are compared to experimental estimates from the SBH17 dataset~\cite{tchakouaSBH17BenchmarkDatabase2023}. All values are in kJ/mol.",
    float_fmt = '%.1f',
    rotate_column_header=True,
    replace_input= {
        r' & \rotatebox{90}{Experiment}': r'System & \rotatebox{90}{Experiment}',
        r' 0.0 ': r' - '
    },
    filename = "Tables/SI_Table-SBH17_Barrier_Heights_NSC.tex"
)

convert_df_to_latex_input(
    df=sbh17_method_df_2.round(1),
    start_input = '\\begin{table}[h]\n',
    label = "tab:sbh17_2",
    caption = r"Barrier heights of the SBH17 benchmark set calculated with various DFT functionals. These are compared to experimental estimates from the SBH17 dataset~\cite{tchakouaSBH17BenchmarkDatabase2023}. All values are in kJ/mol.",
    float_fmt = '%.1f',
    rotate_column_header=True,
    replace_input= {
        r' & \rotatebox{90}{PBE}': r'System & \rotatebox{90}{PBE}',
        r' 0.0 ': r' - '
    },
    adjustbox=1,
    filename = "Tables/SI_Table-SBH17_Barrier_Heights_DFT.tex"
)

# Change column names to labels
sbh17_method_df.columns = [method_dict[method]['label'] if method in method_dict else method for method in sbh17_method_df.columns.tolist()]



display(sbh17_method_df)



,Experiment,dhBEEF-vdW,hBEEF-vdW,RPA@PBE,BEEF-vdW,PBE,PBEsol,RPBE,revPBE,BLYP,r$^2$SCAN,MS2,revTPSS,PBE-D3,PBE-DdsC,RPBE-D3,optPBE-vdW,rev-vdW-DF2,r$^2$SCAN+rVV10
\ce{H2 + Cu(111)},60.6,84.2,91.2,64.6,93.2,41.7,0.7,73.3,68.9,93.0,31.2,30.8,50.1,20.5,35.1,8.3,69.2,45.9,24.1
\ce{H2 + Cu(100)},71.4,94.3,102.3,70.6,100.4,55.8,13.1,88.4,83.8,108.9,46.6,50.1,63.1,32.8,49.1,21.6,84.0,59.7,39.1
\ce{H2 + Cu(110)},76.1,110.9,122.6,81.8,118.6,63.2,14.6,99.7,94.1,118.0,50.6,53.4,75.2,39.4,55.7,32.5,95.1,67.0,43.7
\ce{H2 + Pt(111)},-0.8,3.9,3.0,5.0,10.9,1.6,-14.4,17.5,16.7,20.7,-4.3,-7.0,2.6,-11.2,-6.6,-18.7,2.8,-4.1,-10.2
\ce{H2 + Pt(211)},-8.0,-5.7,-5.3,-2.8,-3.2,-2.7,-7.2,3.8,4.5,4.4,-5.3,-6.8,-1.3,-10.4,-8.3,-12.3,-7.0,-6.5,-9.1
\ce{H2 + Ru(0001)},0.4,-1.8,-0.3,3.1,2.0,0.7,-5.6,9.0,9.6,11.1,-2.5,-4.3,0.8,-7.5,-5.8,-10.1,-2.7,-3.7,-6.8
\ce{H2 + Ni(111)},2.3,4.9,4.4,10.7,11.8,2.0,-12.2,17.3,16.6,20.0,-2.3,-29.7,0.9,-15.6,-7.2,-15.6,2.6,-1.9,-8.1
\ce{H2 + Ag(111)},104.4,157.1,166.0,136.0,164.1,106.2,61.0,140.4,135.3,158.7,106.2,99.2,112.8,94.4,99.9,77.3,132.5,107.0,97.9
\ce{N2 + Ru(0001)},177.5,154.9,175.2,109.3,164.7,131.5,63.1,178.5,171.9,218.1,120.7,128.3,98.5,105.8,124.0,40.6,133.2,97.3,100.9
\ce{N2 + Ru(10-10)},38.6,-0.7,19.3,-62.4,23.3,-22.5,-106.8,36.9,27.7,77.1,-48.0,-36.8,-58.5,-50.8,-32.6,-130.6,-19.2,-64.1,-72.3


In [124]:
# Plot of W-EA against DFT methods

fig, axs = plt.subplots(3,1, figsize=(6.65, 7), sharey=True, sharex=True, dpi=450, constrained_layout = True)

# Plot dhBEEFvdW, hBEEFvdW, RPA, BEEFvdW
for method in ['dhBEEFvdW', 'hBEEFvdW', 'RPA', 'BEEFvdW']:
    axs[0].scatter(list(sbh17_df['W-EA Order']),[sbh17_method_dict[system][method] - sbh17_method_dict[system]['Experiment'] for system in list(sbh17_df['W-EA'].index)], label=f"{method_dict[method]['label']} ({np.mean(np.abs([sbh17_method_dict[system][method] - sbh17_method_dict[system]['Experiment'] for system in list(sbh17_df['W-EA'].index)])):.1f})",edgecolor='black', s=30, zorder=3,linewidth=1)
axs[0].axhline(0, color='black', linestyle='--', zorder=1)

# Plot DFT XC functionals
for method in dft_xc_list[:8]:
    axs[1].scatter(list(sbh17_df['W-EA Order']),[sbh17_method_dict[system][method] - sbh17_method_dict[system]['Experiment'] for system in list(sbh17_df['W-EA'].index)], label=f"{method_dict[method]['label']} ({np.mean(np.abs([sbh17_method_dict[system][method] - sbh17_method_dict[system]['Experiment'] for system in list(sbh17_df['W-EA'].index)])):.1f})",edgecolor='black', s=30, zorder=3,linewidth=1)
axs[1].axhline(0, color='black', linestyle='--', zorder=1)

for method in dft_xc_list[8:]:
    axs[2].scatter(list(sbh17_df['W-EA Order']),[sbh17_method_dict[system][method] - sbh17_method_dict[system]['Experiment'] for system in list(sbh17_df['W-EA'].index)], label=f"{method_dict[method]['label']} ({np.mean(np.abs([sbh17_method_dict[system][method] - sbh17_method_dict[system]['Experiment'] for system in list(sbh17_df['W-EA'].index)])):.1f})",edgecolor='black', s=30, zorder=3,linewidth=1)
axs[2].axhline(0, color='black', linestyle='--', zorder=1)


axs[2].set_xlabel('Increasing W-EA (eV)')
axs[2].set_xticks(sbh17_df['W-EA Order'])
axs[2].set_xticklabels([f"{sbh17_df.loc[system, 'system']} ({w_ea:.1f})" for system, w_ea in sbh17_df['W-EA'].items()], rotation=45, ha='right')

axs[0].set_ylabel('Error vs. Experiment (kJ/mol)')
axs[1].set_ylabel('Error vs. Experiment (kJ/mol)')
axs[2].set_ylabel('Error vs. Experiment (kJ/mol)')

axs[0].legend(loc='upper left', bbox_to_anchor=(1, 1))
axs[1].legend(loc='upper left', bbox_to_anchor=(1, 1))
axs[2].legend(loc='upper left', bbox_to_anchor=(1, 1))

axs[0].set_ylim([-200,100])


plt.savefig('Figures/SI_Figure-SBH17_W_EA_vs_Methods.png')

In [125]:
fig, ax = plt.subplots(3,3,figsize=(6.69,6.15), dpi=600, constrained_layout=True, gridspec_kw={'height_ratios': [2,2.2,2]})

ax[0,0].remove()
ax[0,1].remove()
ax[0,2].remove()

# Combine ax[0,0] and ax[1,0] and ax[2,0] into one large plot for Experiment vs RPA
ax_exp_rpa = plt.subplot2grid((3, 3), (0, 0), colspan=3)

method_list = ['RPA','dhBEEFvdW','hBEEFvdW','BEEFvdW', '14_r2SCANrVV10', '07_MS2', '03_RPBE','01_PBE'][::-1]

# Plot the MSE for H2, CH4, N2
x_mad = np.arange(len(method_list))
bar_width = 0.3

for method_idx, method in enumerate(method_list):
    mad = [sbh17_method_df.loc['MAD (Total)', method_dict[method]['label']], sbh17_method_df.loc['MAD (CH4)', method_dict[method]['label']], sbh17_method_df.loc['MAD (H2)', method_dict[method]['label']],  sbh17_method_df.loc['MAD (N2)', method_dict[method]['label']]]
    if method in ['dhBEEFvdW','hBEEFvdW']:
        ax_exp_rpa.bar(np.arange(4)*4+method_idx*0.4, mad,  width=bar_width, label=method_dict[method]['label'], color = color_dict[method_dict[method]['color']],edgecolor='black', linewidth=1)
    else:
        ax_exp_rpa.bar(np.arange(4)*4+method_idx*0.4, mad,  width=bar_width, label=method_dict[method]['label'], color = color_dict[method_dict[method]['color']])

ax_exp_rpa.set_xticks(np.arange(4)*4 + (len(method_list)-1)*0.4/2)
ax_exp_rpa.set_xticklabels([r'\textbf{Total}',r'\textbf{CH\textsubscript{4}@Metal}', r'\textbf{H\textsubscript{2}@Metal}', r'\textbf{N\textsubscript{2}@Metal}'], fontsize=9)

ax_exp_rpa.set_ylabel('Mean Absolute Deviation (kJ/mol)', fontsize=9)

ymax = 50
group_idx_n2 = 3
x_base_n2 = group_idx_n2 * 4

arrow_methods = {'01_PBE', '14_r2SCANrVV10', 'RPA', '07_MS2'}

for method_idx, method in enumerate(method_list):
    if method not in arrow_methods:
        continue

    y = sbh17_method_df.loc['MAD (N2)', method_dict[method]['label']]
    if y <= ymax:
        continue

    x = x_base_n2 + method_idx * 0.4

    # --- DRAW THE ARROW (this was missing) ---
    ax_exp_rpa.annotate(
        "",
        xy=(x, ymax),        # arrow head
        xytext=(x, ymax - 10),  # arrow tail (17 → 20)
        arrowprops=dict(
            arrowstyle='-|>',
            lw=0.9,
            color='black'
        ),
        annotation_clip=False
    )

    # --- numeric label above arrow ---
    ax_exp_rpa.text(
        x+0.03, ymax - 17,
        f"{y:.1f}",
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold',
        rotation=90,
        clip_on=False
    )


ax_exp_rpa.legend(frameon=False,ncol=4)

# Move y-axis to the right
ax_exp_rpa.set_ylim([0,50])

# Remove top and right spines
ax_exp_rpa.spines['top'].set_visible(False)
ax_exp_rpa.spines['right'].set_visible(False)

# Remove axes for axs[1,1] and axs[1,2]
ax[1,1].axis('off')
ax[1,2].axis('off')
ax[1,0].axis('off')

# Add BH76 MAD to axs[2,1]
for method_idx, method in enumerate(method_list):
    mad = nbh56_eint_df.loc['MAD', method_dict[method]['label']]
    if method in ['dhBEEFvdW','hBEEFvdW']:
        ax[2,0].bar(method_idx, mad, width=0.6, label=method_dict[method]['label'],color=color_dict[method_dict[method]['color']],edgecolor='black', linewidth=1)
    else:
        ax[2,0].bar(method_idx, mad, width=0.6, label=method_dict[method]['label'],color=color_dict[method_dict[method]['color']])

# Add S66 MAD to axs[2,0]
for method_idx, method in enumerate(method_list):
    mad = s66_eint_df.loc['MAD', method_dict[method]['label']]
    if method in ['dhBEEFvdW','hBEEFvdW']:
        ax[2,1].bar(method_idx, mad, width=0.6, label=method_dict[method]['label'],color=color_dict[method_dict[method]['color']],edgecolor='black', linewidth=1)
    else:
        ax[2,1].bar(method_idx, mad, width=0.6, label=method_dict[method]['label'],color=color_dict[method_dict[method]['color']])

# Add SE20 to axs[2,2]
for method_idx, method in enumerate(method_list):
    mad = se20_eint_df.loc['MAD', method_dict[method]['label']]
    if method in ['dhBEEFvdW','hBEEFvdW']:
        ax[2,2].bar(method_idx, mad, width=0.6, label=method_dict[method]['label'],color=color_dict[method_dict[method]['color']],edgecolor='black', linewidth=1)
    else:
        ax[2,2].bar(method_idx, mad, width=0.6, label=method_dict[method]['label'],color=color_dict[method_dict[method]['color']])

# Move xlabel to the top for the three MAD plots
ax[2,0].xaxis.set_label_position('top')
ax[2,1].xaxis.set_label_position('top')
ax[2,2].xaxis.set_label_position('top')

# Remove right and bottom spines for the three MAD plots
for i in range(3):
    ax[2,i].spines['right'].set_visible(False)
    ax[2,i].spines['top'].set_visible(False)

ax[2,0].set_ylabel('Mean Absolute Deviation (kJ/mol)', fontsize=9)

ax[2,0].set_ylim([0,60])
ax[2,1].set_ylim([0,20])
ax[2,2].set_ylim([0,0.6])

ax[2,0].set_xticks([])
ax[2,1].set_xticks([])
ax[2,2].set_xticks([])

ax[2,0].set_xlabel('Gas Barrier Heights (NBH56)', fontsize=9)
ax[2,1].set_xlabel('Non-Covalent Interactions (S66)', fontsize=9)
ax[2,2].set_xlabel(r'Surface Energies in J/m$^2$ (SE20)', fontsize=9)

plt.savefig('Figures/MAIN_Figure-SBH17.png')